# AkutuStocks | Gathering sampai Dataset Siap Pakai (Colab)
**Peran:** Rini (Data Science) | **Tim:** A26-CS047

Notebook ini mengolah **tiga dataset yang sudah dikumpulkan** (`retail_sales`, `retail_inventory`, `flipkart`) dari awal sampai menjadi dataset yang siap dipakai tiap role. Produk aplikasi terdiri dari produk **retail** (sepatu dan pakaian anak, stok dan penjualan dari data nyata) ditambah **33 produk kaos kaki asli dari Flipkart** (nama, gambar, harga dari Flipkart; **stok dan penjualan sintetis**).

| Tahap | Hasil |
|---|---|
| 1-2 Gathering | profil tiga dataset |
| 3 Assessing | `02_assessing/temuan_assessing.csv` (masalah kualitas, jumlah, dampak, penanganan) |
| 4 Cleaning | `03_cleaning/catatan_cleaning.csv` (setiap langkah dengan jumlah baris sebelum dan sesudah) |
| 5-7 Integrasi ke kolom `TEMPLATE_BARANG` | master barang, stok per gudang, riwayat, berkas Kasir Pintar, skenario uji |
| 8 | **Role AI:** dataset intent berlabel + data uji di luar konteks |
| 9 Validasi | pemeriksaan otomatis (harus LULUS semua) |
| 10-11 Ekspor | **Role Full-Stack:** CSV + SQL; **Analisis:** data lengkap; README dan ZIP |

**Cara pakai:** Runtime > Run all. Saat diminta, unggah 3 berkas CSV. Parameter ada di sel 1. Baca asumsi dan keterbatasan di `README.md` hasil keluaran.

In [1]:
# ============ 1. SETUP, PARAMETER, DAN LOKASI BERKAS ============
import os, re, sys, json, glob, shutil, subprocess, math
from collections import defaultdict
from datetime import date, datetime, timedelta, timezone
import numpy as np
import pandas as pd

try:
    import bcrypt
except ImportError:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "bcrypt"]); import bcrypt
    except Exception:
        bcrypt = None

# ---- Parameter (boleh diubah) ----
SEED             = 42
N_SKU_APP        = 300      # jumlah produk untuk dataset aplikasi (database)
HARGA_SKALA      = 1000     # ASUMSI: harga dataset dikali 1.000 agar bermagnitudo rupiah (bukan harga nyata)
MIN_UNIT_TERJUAL = 10       # produk calon aplikasi: minimal terjual sekian unit selama periode
LEAD_WEEKS       = 2        # minimum_stok = kebutuhan penjualan mingguan x LEAD_WEEKS
KURS_INR_IDR     = 190      # ASUMSI kasar: Rp per 1 rupee India (untuk harga kaos kaki Flipkart)
SOCK_RATE_MEDIAN = 0.5      # SINTETIS: median penjualan kaos kaki (unit/hari, semua gudang)
SOCK_RETUR_PROB  = 0.04     # SINTETIS: peluang retur pada hari ada penjualan kaos kaki
HARI_PENDING     = 3        # aktivitas pada N hari terakhir masih berstatus Diajukan
TARGET_PER_INTENT = 100     # jumlah kalimat per kelas intent
GUDANG_NAMA      = ["Gudang A", "Gudang B", "Gudang C"]
DEMO_PASSWORD    = "Demo#2026"   # HANYA untuk akun contoh
OUT_DIR          = "akutustocks_dataset"

WIB = timezone(timedelta(hours=7))
rng = np.random.default_rng(SEED)
for sub in ["01_gathering", "02_assessing", "03_cleaning", "fs/csv", "fs/sql", "fs/kasir_pintar", "fs/skenario_uji", "ai", "analisis", "dokumentasi"]:
    os.makedirs(f"{OUT_DIR}/{sub}", exist_ok=True)

def cari_berkas(kata):
    for p in glob.glob("*.csv") + glob.glob("/content/*.csv"):
        if kata in os.path.basename(p).lower(): return p
    return None
PATH = {k: cari_berkas(k) for k in ("sales", "inventory", "flipkart")}
if any(v is None for v in PATH.values()):
    try:
        from google.colab import files
        print("Unggah 3 berkas: retail_sales..., retail_inventory..., flipkart...")
        files.upload(); PATH = {k: cari_berkas(k) for k in ("sales", "inventory", "flipkart")}
    except ImportError:
        pass
assert all(PATH.values()), f"Berkas belum lengkap: {PATH}"
print("Berkas ditemukan:", PATH)

Unggah 3 berkas: retail_sales..., retail_inventory..., flipkart...


Saving flipkart_com-ecommerce_sample.csv to flipkart_com-ecommerce_sample.csv
Saving retail_sales_ml_apl (1).csv to retail_sales_ml_apl (1).csv
Saving retail_inventory_ml_apl (1).csv to retail_inventory_ml_apl (1).csv
Berkas ditemukan: {'sales': 'retail_sales_ml_apl (1).csv', 'inventory': 'retail_inventory_ml_apl (1).csv', 'flipkart': 'flipkart_com-ecommerce_sample.csv'}


## 2. Gathering
Memuat tiga dataset dan mencatat profilnya. Sales dan inventory memiliki kunci yang sama (`Product No`, `Store`); Flipkart tidak punya kunci ke keduanya.

In [2]:
# ============ 2. GATHERING ============
sales_raw = pd.read_csv(PATH["sales"])
inv_raw   = pd.read_csv(PATH["inventory"])
fk_raw    = pd.read_csv(PATH["flipkart"])

def rentang(d, kol):
    t = pd.to_datetime(d[kol], errors="coerce"); t = t[t.dt.year < 3000]
    return f"{t.min().date()} s.d. {t.max().date()}"
PROFIL = pd.DataFrame([
    dict(dataset="retail_sales", baris=len(sales_raw), kolom=sales_raw.shape[1], rentang_tanggal=rentang(sales_raw, "Transaction Date"),
         kunci="Product No, Store", peran="barang keluar/retur (hanya kuantitas)"),
    dict(dataset="retail_inventory", baris=len(inv_raw), kolom=inv_raw.shape[1], rentang_tanggal=rentang(inv_raw, "Start Date"),
         kunci="Product No, Store", peran="master barang, harga, stok per toko"),
    dict(dataset="flipkart", baris=len(fk_raw), kolom=fk_raw.shape[1], rentang_tanggal=rentang(fk_raw, "crawl_timestamp"),
         kunci="pid (tidak ada kunci ke dua dataset lain)", peran="acuan gambar, keterangan, berat"),
])
PROFIL.to_csv(f"{OUT_DIR}/01_gathering/profil_dataset.csv", index=False)
print(PROFIL.to_string(index=False))
print("Kunci bersama sales-inventory -> produk:", sales_raw['Product No'].nunique(), "| toko:", sales_raw['Store'].nunique())

         dataset  baris  kolom            rentang_tanggal                                     kunci                                 peran
    retail_sales 125751     17 2025-06-01 s.d. 2026-04-24                         Product No, Store barang keluar/retur (hanya kuantitas)
retail_inventory 284755     18 2025-06-01 s.d. 2026-04-24                         Product No, Store   master barang, harga, stok per toko
        flipkart  20000     15 2015-12-01 s.d. 2016-06-28 pid (tidak ada kunci ke dua dataset lain)       acuan gambar, keterangan, berat
Kunci bersama sales-inventory -> produk: 2326 | toko: 40


## 3. Assessing
Menilai kualitas data sebelum dibersihkan: kelengkapan, validitas, konsistensi, keunikan, integritas, dan relevansi. Termasuk temuan bahwa pencarian kata "sock" di Flipkart ikut menangkap produk *Socket Holder*.

In [3]:
# ============ 3. ASSESSING (menilai kualitas sebelum dibersihkan) ============
TEMUAN = []
def temuan(dataset, aspek, pemeriksaan, jumlah, total, dampak, penanganan):
    TEMUAN.append(dict(dataset=dataset, aspek=aspek, pemeriksaan=pemeriksaan, jumlah=int(jumlah), dari=int(total),
                       persen=round(100 * jumlah / total, 2) if total else None, dampak=dampak, penanganan=penanganan))

S, I, F = sales_raw, inv_raw, fk_raw
ext_s = S["Product Description"].str.extract(r"^(PROD-\d+)\s+(SEG-\d+)\s+(.*)$")
ext_i = I["Product Description"].str.extract(r"^(PROD-\d+)\s+(SEG-\d+)\s+(.*)$")
pair_i = set(zip(I["Product No"], I["Store"]))
pair_s = set(zip(S["Product No"], S["Store"]))
# sales
temuan("sales", "kelengkapan", "Reason of Return kosong", S["Reason of Return"].isna().sum(), len(S), "Kolom tidak berguna", "Kolom dibuang")
temuan("sales", "keunikan", "Sales Channel bernilai tunggal", len(S) if S["Sales Channel"].nunique() == 1 else 0, len(S), "Tidak informatif", "Kolom dibuang")
temuan("sales", "validitas", "Qty Sold negatif (retur)", (S["Qty Sold"] < 0).sum(), len(S), "Bisa salah dibaca sebagai barang keluar negatif", "Diubah menjadi jenis 'retur' dengan qty positif")
temuan("sales", "validitas", "Qty Sold = 0", (S["Qty Sold"] == 0).sum(), len(S), "Transaksi tanpa pergerakan stok", "Baris dibuang")
temuan("sales", "konsistensi", "Is Return tidak sejalan dengan tanda Qty", ((S["Is Return"] == 1) != (S["Qty Sold"] < 0)).sum(), len(S), "Salah klasifikasi", "Diperiksa; tidak ada tindakan bila 0")
temuan("sales", "keunikan", "Baris duplikat penuh", S.duplicated().sum(), len(S), "Penjualan terhitung ganda", "Dibuang bila ada")
temuan("sales", "konsistensi", "Kode PROD- di Product Description berbeda dengan Product No", (ext_s[0] != S["Product No"]).sum(), len(S), "Identitas barang ganda/membingungkan", "Kode di deskripsi dibuang dari nama barang")
temuan("sales", "integritas", "Pasangan produk-toko penjualan tanpa baris inventory", len(pair_s - pair_i), len(pair_s), "Stok awal tidak diketahui", "Stok awal diestimasi dari rekonsiliasi")
temuan("sales", "kepatuhan plan", "Kolom keuangan (Sales Amount, Cogs) tersedia", len(S), len(S), "Plan melarang analisis keuangan/transaksi", "Kolom keuangan tidak dipakai dan tidak diekspor")
# inventory
temuan("inventory", "validitas", "Qty on hand negatif", (I["Qty on hand"] < 0).sum(), len(I), "Stok tidak mungkin negatif; indikasi salah input", "Dikoreksi menjadi 0, nilai asli disimpan (qty_mentah)")
temuan("inventory", "validitas", "Qty on hand = 0", (I["Qty on hand"] == 0).sum(), len(I), "Baris tanpa stok", "Dipertahankan sebagai stok 0")
temuan("inventory", "validitas", "Harga jual atau biaya <= 0", ((I["Stock Unit Selling Price"] <= 0) | (I["Stock Unit Cost Price"] <= 0)).sum(), len(I), "Harga tidak bisa dipakai", "Diubah menjadi kosong, harga diambil dari baris lain")
temuan("inventory", "validitas", "Biaya >= harga jual pada baris berstok", ((I["Stock Unit Cost Price"] >= I["Stock Unit Selling Price"]) & (I["Qty on hand"] > 0)).sum(), len(I), "Melanggar aturan template (beli < jual)", "Harga beli diimputasi dari rasio median kategori, ditandai")
temuan("inventory", "konsistensi", "End Date bernilai penanda 9999-12-31", (I["End Date"] == "9999-12-31").sum(), len(I), "Tanggal palsu bila dibaca langsung", "Dianggap snapshot aktif, tanggal akhir dikosongkan")
temuan("inventory", "validitas", "Start Date lebih besar dari End Date", (pd.to_datetime(I["Start Date"]) > pd.to_datetime(I["End Date"], errors="coerce")).sum(), len(I), "Periode tidak masuk akal", "Tidak ada bila 0")
gp = I.groupby("Product No")["Stock Unit Selling Price"].nunique()
temuan("inventory", "konsistensi", "Produk dengan lebih dari satu harga jual", (gp > 1).sum(), len(gp), "Harga master ambigu (efek promo/markdown)", "Harga master = median status Full Price")
temuan("inventory", "konsistensi", "Kode PROD- di Product Description berbeda dengan Product No", (ext_i[0] != I["Product No"]).sum(), len(I), "Identitas barang ganda", "Dibuang dari nama barang")
temuan("inventory", "keunikan", "Baris duplikat penuh", I.duplicated().sum(), len(I), "Stok terhitung ganda", "Dibuang bila ada")
# flipkart
temuan("flipkart", "kelengkapan", "retail_price kosong", F["retail_price"].isna().sum(), len(F), "Harga tidak dipakai di proyek ini", "Tidak dipakai")
temuan("flipkart", "kelengkapan", "brand kosong", F["brand"].isna().sum(), len(F), "Tidak dipakai", "Tidak dipakai")
temuan("flipkart", "kelengkapan", "image kosong", (F["image"].isna() | (F["image"] == "[]")).sum(), len(F), "Tidak ada gambar", "Baris dikeluarkan dari kolam acuan")
temuan("flipkart", "keunikan", "pid duplikat", F["pid"].duplicated().sum(), len(F), "Produk ganda", "Dihapus, sisakan satu")
temuan("flipkart", "keunikan", "product_name duplikat", F["product_name"].duplicated().sum(), len(F), "Nama sama untuk varian berbeda", "Dibiarkan; pid menjadi identitas")
temuan("flipkart", "validitas", "product_name lebih dari 80 karakter", (F["product_name"].str.len() > 80).sum(), len(F), "Melebihi batas template", "Nama Flipkart tidak dipakai sebagai nama barang")
temuan("flipkart", "relevansi", "Kategori bukan alas kaki/pakaian anak", (~F["product_category_tree"].str.contains("Footwear|Kids", regex=True, na=False)).sum(), len(F), "Tidak relevan dengan data retail", "Kolam acuan dibatasi per kelompok kategori")
_naive = F["product_name"].str.contains("sock", case=False, na=False) | F["product_category_tree"].str.contains("sock", case=False, na=False)
_tepat = F["product_name"].str.contains(r"\bsocks?\b", case=False, regex=True, na=False)
temuan("flipkart", "relevansi", "Pencarian 'sock' menangkap produk non-kaos kaki (mis. Phone Socket Holder)", (_naive & ~_tepat).sum(), _naive.sum(), "Produk bukan kaos kaki ikut terpilih", "Pencocokan kata utuh (socks) pada nama produk")
temuan("lintas dataset", "relevansi", "Tema retail: sepatu dan pakaian anak (bukan kaos kaki)", len(set(I['Product No'])), len(set(I['Product No'])), "Domain berbeda dari Akutusocks", "Ditulis sebagai keterbatasan di README")
temuan("lintas dataset", "integritas", "Flipkart tidak punya kunci ke retail", len(F), len(F), "Tidak bisa digabung dengan join", "Dipasangkan per kelompok kategori, diberi label 'acuan acak'")
DF_TEMUAN = pd.DataFrame(TEMUAN); DF_TEMUAN.to_csv(f"{OUT_DIR}/02_assessing/temuan_assessing.csv", index=False)
print(DF_TEMUAN[["dataset", "pemeriksaan", "jumlah", "dari", "persen"]].to_string(index=False))

       dataset                                                                pemeriksaan  jumlah   dari  persen
         sales                                                    Reason of Return kosong  125751 125751  100.00
         sales                                             Sales Channel bernilai tunggal  125751 125751  100.00
         sales                                                   Qty Sold negatif (retur)    7491 125751    5.96
         sales                                                               Qty Sold = 0       2 125751    0.00
         sales                                   Is Return tidak sejalan dengan tanda Qty       0 125751    0.00
         sales                                                       Baris duplikat penuh       0 125751    0.00
         sales                Kode PROD- di Product Description berbeda dengan Product No  125751 125751  100.00
         sales                       Pasangan produk-toko penjualan tanpa baris inventory    258

## 4. Cleaning
Setiap langkah dicatat ke `catatan_cleaning.csv`. Kolom keuangan penjualan tidak dipakai (batasan project plan). Stok negatif dikoreksi ke 0 dengan nilai asli disimpan di `qty_mentah`.

In [4]:
# ============ 4. CLEANING (tiap langkah dicatat) ============
LOG = []
def catat_log(dataset, aksi, sebelum, sesudah, ket=""):
    LOG.append(dict(dataset=dataset, aksi=aksi, baris_sebelum=int(sebelum), baris_sesudah=int(sesudah), keterangan=ket))

# ---------- SALES ----------
s = sales_raw.copy(); n = len(s)
s = s.drop_duplicates(); catat_log("sales", "Hapus baris duplikat", n, len(s)); n = len(s)
s = s[s["Qty Sold"] != 0]; catat_log("sales", "Hapus baris Qty Sold = 0", n, len(s), "tidak ada pergerakan stok")
s = s[["Transaction Date", "Sales Type", "Is Return", "Product No", "Store", "Qty Sold"]]
catat_log("sales", "Buang kolom", len(s), len(s), "Reason of Return (kosong 100%), Sales Channel (tunggal), Sales Amount, Cogs & kolom keuangan lain (dilarang plan)")
s["tanggal"] = pd.to_datetime(s["Transaction Date"])
s["jenis"] = np.where(s["Is Return"] == 1, "retur", "keluar"); s["qty"] = s["Qty Sold"].abs().astype(int)
catat_log("sales", "Retur: Qty negatif -> jenis 'retur' dengan qty positif", len(s), len(s), f"{(s['jenis']=='retur').sum()} baris retur")
PENJUALAN = (s.groupby(["tanggal", "Product No", "Store", "jenis"], as_index=False)
              .agg(qty=("qty", "sum"), tipe_penjualan=("Sales Type", lambda x: x.mode().iat[0]))
              .rename(columns={"Product No": "kode_barang", "Store": "toko"}).assign(sumber="retail"))
catat_log("sales", "Agregasi per (tanggal, produk, toko, jenis)", len(s), len(PENJUALAN))
TGL_MIN, TGL_MAX = PENJUALAN["tanggal"].min().date(), PENJUALAN["tanggal"].max().date()
N_HARI = (TGL_MAX - TGL_MIN).days + 1

# ---------- INVENTORY ----------
v = inv_raw.copy(); n = len(v)
v = v.drop_duplicates(); catat_log("inventory", "Hapus baris duplikat", n, len(v))
ex = v["Product Description"].str.extract(r"^(PROD-\d+)\s+(SEG-\d+)\s+(.*)$")
v["kode_segmen"], v["nama_segmen"] = ex[1], ex[2]
catat_log("inventory", "Pisahkan Product Description", len(v), len(v), f"kode PROD- di deskripsi dibuang; gagal parse: {int(ex[0].isna().sum())} baris")
v["aktif"] = v["End Date"] == "9999-12-31"
v["tgl_mulai"] = pd.to_datetime(v["Start Date"])
catat_log("inventory", "Penanda 9999-12-31 -> snapshot aktif", len(v), len(v), f"{int(v['aktif'].sum())} baris aktif; tanggal akhir dikosongkan")
v["qty_mentah"] = v["Qty on hand"]; v["qty"] = v["qty_mentah"].clip(lower=0)
catat_log("inventory", "Qty negatif -> 0 (nilai asli di qty_mentah)", len(v), len(v), f"{int((v['qty_mentah']<0).sum())} baris dikoreksi")
v["harga_jual_unit"] = v["Stock Unit Selling Price"].where(v["Stock Unit Selling Price"] > 0)
v["harga_beli_unit"] = v["Stock Unit Cost Price"].where(v["Stock Unit Cost Price"] > 0)
catat_log("inventory", "Harga <= 0 -> kosong", len(v), len(v), f"jual {int(v['harga_jual_unit'].isna().sum())}, beli {int(v['harga_beli_unit'].isna().sum())} baris")

# master produk
base = (v.sort_values("tgl_mulai").groupby("Product No")
         .agg(divisi=("Product Division", "first"), kategori=("Product Category", "first"), subkategori=("Product Subcategory", "first"),
              supplier=("Supplier", "first"), kode_segmen=("kode_segmen", "first"), nama_segmen=("nama_segmen", "first")))
fp = v[v["Stock Status"] == "Full Price"]
MASTER = base.copy()
MASTER["harga_jual_ref"] = fp.groupby("Product No")["harga_jual_unit"].median().reindex(MASTER.index).fillna(v.groupby("Product No")["harga_jual_unit"].median())
MASTER["harga_beli_ref"] = fp.groupby("Product No")["harga_beli_unit"].median().reindex(MASTER.index).fillna(v.groupby("Product No")["harga_beli_unit"].median())
n0 = len(MASTER); MASTER = MASTER[MASTER["harga_jual_ref"].notna()]
catat_log("inventory", "Produk tanpa harga jual valid dikeluarkan dari master", n0, len(MASTER))
valid = fp[(fp["harga_beli_unit"] < fp["harga_jual_unit"])]
_tmp = valid.merge(MASTER[["kategori"]].rename(columns={"kategori": "kat"}), left_on="Product No", right_index=True)
rasio_kat = (_tmp["harga_beli_unit"] / _tmp["harga_jual_unit"]).groupby(_tmp["kat"]).median()
rasio_all = float((_tmp["harga_beli_unit"] / _tmp["harga_jual_unit"]).median())
bad = MASTER["harga_beli_ref"].isna() | (MASTER["harga_beli_ref"] >= MASTER["harga_jual_ref"])
MASTER["harga_beli_diimputasi"] = bad
MASTER.loc[bad, "harga_beli_ref"] = (MASTER.loc[bad, "harga_jual_ref"] * MASTER.loc[bad, "kategori"].map(rasio_kat).fillna(rasio_all)).round(2)
catat_log("inventory", "Harga beli >= jual atau kosong -> imputasi rasio median kategori", len(MASTER), len(MASTER), f"{int(bad.sum())} produk diimputasi (kolom harga_beli_diimputasi)")
assert (MASTER["harga_beli_ref"] < MASTER["harga_jual_ref"]).all()

# nama barang unik (maks 80 karakter)
MASTER["nama_barang"] = (MASTER["nama_segmen"].fillna(MASTER["subkategori"]) + " " + MASTER["kode_segmen"].fillna("")).str.strip().str.slice(0, 80)
dup = MASTER["nama_barang"].duplicated(keep=False)
MASTER.loc[dup, "nama_barang"] = MASTER.loc[dup].apply(lambda r: f"{r['nama_barang'][:72]} #{r.name[-5:]}", axis=1)
assert MASTER["nama_barang"].is_unique
catat_log("inventory", "Susun nama_barang = nama segmen + kode segmen (unik, maks 80)", len(MASTER), len(MASTER), f"{int(dup.sum())} nama bentrok diberi akhiran kode")

# stok aktif per (produk, toko)
AKTIF = v[v["aktif"]].groupby(["Product No", "Store"], as_index=False).agg(qty=("qty", "sum"), qty_mentah=("qty_mentah", "sum"), tgl_cek=("tgl_mulai", "max"))
AKTIF = AKTIF.rename(columns={"Product No": "kode_barang", "Store": "toko"})
catat_log("inventory", "Ambil snapshot aktif per (produk, toko)", len(v), len(AKTIF), f"{AKTIF['kode_barang'].nunique()} produk, {AKTIF['toko'].nunique()} toko")

# diskon dari status stok aktif
act = v[v["aktif"] & (v["qty"] > 0) & v["harga_jual_unit"].notna()].merge(MASTER[["harga_jual_ref"]], left_on="Product No", right_index=True)
act["md"] = act["Stock Status"] != "Full Price"
act["disk_row"] = (1 - act["harga_jual_unit"] / act["harga_jual_ref"]).clip(0, 0.6)
q_md = (act["qty"] * act["md"]).groupby(act["Product No"]).sum(); q_all = act["qty"].groupby(act["Product No"]).sum()
med_d = act[act["md"]].groupby("Product No")["disk_row"].median()
share = (q_md / q_all).reindex(MASTER.index).fillna(0)
MASTER["diskon_persen"] = np.where(share >= 0.5, (med_d.reindex(MASTER.index).fillna(0) * 100 / 5).round() * 5, 0.0)

# ringkasan stok dan penjualan per produk
stk = AKTIF.groupby("kode_barang").agg(stok_total_bersih=("qty", "sum"), stok_total_mentah=("qty_mentah", "sum"), n_toko_aktif=("toko", "nunique"))
MASTER = MASTER.join(stk).fillna({"stok_total_bersih": 0, "stok_total_mentah": 0, "n_toko_aktif": 0})
klr = PENJUALAN[PENJUALAN["jenis"] == "keluar"].groupby("kode_barang")["qty"].sum(); rtr = PENJUALAN[PENJUALAN["jenis"] == "retur"].groupby("kode_barang")["qty"].sum()
MASTER["unit_keluar"] = klr.reindex(MASTER.index).fillna(0).astype(int); MASTER["unit_retur"] = rtr.reindex(MASTER.index).fillna(0).astype(int)
MASTER["minimum_stok"] = np.clip(np.ceil(MASTER["unit_keluar"] / (N_HARI / 7) * LEAD_WEEKS), 3, 50).astype(int)
MASTER["punya_stok_aktif"] = MASTER["n_toko_aktif"] > 0
MASTER["sumber_data"] = "retail_inventory+flipkart_acuan"
MASTER.index.name = "kode_barang"

# ---------- FLIPKART (hanya sebagai acuan gambar/keterangan/berat) ----------
fk = fk_raw.drop_duplicates("pid"); catat_log("flipkart", "Hapus pid duplikat", len(fk_raw), len(fk))
def img1(x):
    try: return json.loads(x)[0]
    except Exception: return None
fk = fk.assign(gambar=fk["image"].map(img1))
n = len(fk); fk = fk[fk["gambar"].notna() & fk["product_name"].notna()]; catat_log("flipkart", "Buang baris tanpa gambar/nama", n, len(fk))
t = fk["product_category_tree"]
fk = fk.assign(top=t.str.extract(r'^\["([^>]+?)\s*>>')[0], sec=t.str.extract(r'>>\s*([^>\]"]+?)\s*(?:>>|"\])')[0])
def berat_g(sp):
    m = re.search(r'"key"=>"Weight", "value"=>"([\d.]+)\s*g\b', sp if isinstance(sp, str) else "")
    if not m: return np.nan
    x = float(m.group(1)); return x if 50 <= x <= 1500 else np.nan      # 'kg' & nilai tak wajar = placeholder, diabaikan
fk = fk.assign(berat_gram=fk["product_specifications"].map(berat_g),
               ket=fk["description"].fillna("").str.replace(r"\s+", " ", regex=True).str.slice(0, 150))
POOL = {"Femme Footwear": fk[(fk["top"] == "Footwear") & fk["sec"].str.contains("Women", na=False)],
        "Scholar Footwear": fk[(fk["top"] == "Footwear") & ~fk["sec"].str.contains("Women", na=False)],
        "Junior Apparel": fk[(fk["top"] == "Clothing") & fk["sec"].str.contains("Kids", na=False)]}
catat_log("flipkart", "Kolam acuan per divisi", len(fk), sum(len(p) for p in POOL.values()), str({k: len(p) for k, p in POOL.items()}))
DF_LOG = pd.DataFrame(LOG); DF_LOG.to_csv(f"{OUT_DIR}/03_cleaning/catatan_cleaning.csv", index=False)
print(DF_LOG[["dataset", "aksi", "baris_sebelum", "baris_sesudah"]].to_string(index=False))
print("MASTER:", len(MASTER), "produk |", int(MASTER["punya_stok_aktif"].sum()), "punya stok aktif | periode penjualan", TGL_MIN, "->", TGL_MAX)

  dataset                                                             aksi  baris_sebelum  baris_sesudah
    sales                                             Hapus baris duplikat         125751         125751
    sales                                         Hapus baris Qty Sold = 0         125751         125749
    sales                                                      Buang kolom         125749         125749
    sales           Retur: Qty negatif -> jenis 'retur' dengan qty positif         125749         125749
    sales                      Agregasi per (tanggal, produk, toko, jenis)         125749         125747
inventory                                             Hapus baris duplikat         284755         284755
inventory                                     Pisahkan Product Description         284755         284755
inventory                             Penanda 9999-12-31 -> snapshot aktif         284755         284755
inventory                      Qty negatif -> 0 (nilai 

## 5. Gudang, kaos kaki Flipkart, dan pemilihan produk aplikasi
40 toko dikelompokkan menjadi 3 gudang secara seimbang. **33 produk kaos kaki asli** diambil dari Flipkart dengan pencocokan kata utuh `socks`; harga INR dikonversi dengan kurs asumsi, **stok dan penjualannya sintetis** dan berlabel. Sisa kuota produk diisi produk retail yang dipilih bertingkat per kategori, dengan acuan Flipkart (gambar, keterangan, berat) dipasangkan **acak per kelompok kategori**.

In [5]:
# ============ 5. GUDANG, STOK PER GUDANG, PEMILIHAN PRODUK APLIKASI ============
def at(d, h, m=0, s=0): return datetime(d.year, d.month, d.day, h, m, s, tzinfo=WIB)

# 40 toko -> 3 gudang, seimbang menurut total stok (greedy: toko terbesar ke gudang yang paling kecil)
GUDANG_IDS = list(range(1, len(GUDANG_NAMA) + 1))
stok_toko = AKTIF.groupby("toko")["qty"].sum().sort_values(ascending=False)
beban = {g: 0 for g in GUDANG_IDS}; TOKO_GUDANG = {}
for toko, q in stok_toko.items():
    g = min(beban, key=beban.get); TOKO_GUDANG[toko] = g; beban[g] += int(q)
for toko in set(PENJUALAN["toko"]) - set(TOKO_GUDANG):          # toko yang hanya ada di penjualan
    g = min(beban, key=beban.get); TOKO_GUDANG[toko] = g
DF_TOKO = pd.DataFrame([dict(toko=t, id_gudang=g, nama_gudang=GUDANG_NAMA[g - 1], total_stok_bersih=int(stok_toko.get(t, 0))) for t, g in sorted(TOKO_GUDANG.items())])
GUDANG_SHARE = [beban[g] / sum(beban.values()) for g in GUDANG_IDS]     # porsi gudang mengikuti stok retail
print("Beban stok per gudang:", {GUDANG_NAMA[g - 1]: q for g, q in beban.items()})

AKTIF["id_gudang"] = AKTIF["toko"].map(TOKO_GUDANG)
STOK_G = AKTIF.groupby(["kode_barang", "id_gudang"]).agg(qty=("qty", "sum"), qty_mentah=("qty_mentah", "sum"), tgl_cek=("tgl_cek", "max"))
STOK_F = {k: int(v) for k, v in STOK_G["qty"].items()}
CEK_G = {k: v.date() for k, v in STOK_G["tgl_cek"].items()}

# ---- produk KAOS KAKI asli dari Flipkart (nama, gambar, harga asli); STOK & PENJUALAN SINTETIS ----
SOCKS = fk[fk["product_name"].str.contains(r"\bsocks?\b", case=False, regex=True, na=False) & fk["retail_price"].notna()].copy()
SOCKS["kode_barang"] = "FKS-" + SOCKS["pid"]
assert SOCKS["kode_barang"].str.len().max() <= 20 and SOCKS["kode_barang"].is_unique
_nm = SOCKS["product_name"].str.slice(0, 80); _d = _nm.duplicated(keep=False)
SOCKS["nama_barang"] = _nm.where(~_d, _nm.str.slice(0, 72) + " #" + SOCKS["pid"].str[-5:])
SOCKS["harga_jual_idr"] = (SOCKS["retail_price"] * KURS_INR_IDR / 500).round() * 500
SOCKS["harga_beli_idr"] = np.minimum((SOCKS["harga_jual_idr"] * rasio_all / 500).round() * 500, SOCKS["harga_jual_idr"] - 500)
SOCKS["diskon_persen"] = ((1 - SOCKS["discounted_price"] / SOCKS["retail_price"]).clip(0, 0.6) * 100 / 5).round() * 5
SOCKS = SOCKS.set_index("kode_barang", drop=False)
catat_log("flipkart", "Seleksi kaos kaki (kata utuh 'socks')", len(fk), len(SOCKS), f"{SOCKS['product_name'].nunique()} nama unik, {SOCKS['brand'].nunique()} merek; harga INR x {KURS_INR_IDR} (asumsi); beli diimputasi")
GUD_TOKO = defaultdict(list)
for _t, _g in TOKO_GUDANG.items(): GUD_TOKO[_g].append(_t)
_rows, _lam = [], {}
for kode in SOCKS.index:
    _lam[kode] = lam = float(np.exp(rng.normal(np.log(SOCK_RATE_MEDIAN), 0.6)))
    for g, share in zip(GUDANG_IDS, GUDANG_SHARE):
        n_ev = rng.poisson(lam * share / 1.5, size=N_HARI)
        for di in np.nonzero(n_ev)[0]:
            tgl = pd.Timestamp(TGL_MIN) + pd.Timedelta(days=int(di)); toko = str(rng.choice(GUD_TOKO[g]))
            _rows.append(dict(tanggal=tgl, kode_barang=kode, toko=toko, jenis="keluar", qty=int(n_ev[di] + rng.poisson(0.5 * n_ev[di])), tipe_penjualan="sintetis", sumber="sintetis_kaos_kaki"))
            if rng.random() < SOCK_RETUR_PROB: _rows.append(dict(tanggal=tgl, kode_barang=kode, toko=toko, jenis="retur", qty=1, tipe_penjualan="sintetis", sumber="sintetis_kaos_kaki"))
PENJUALAN_SOCK = pd.DataFrame(_rows)
PENJUALAN = pd.concat([PENJUALAN, PENJUALAN_SOCK], ignore_index=True)
_k = PENJUALAN_SOCK[PENJUALAN_SOCK.jenis == "keluar"].groupby("kode_barang")["qty"].sum(); _r = PENJUALAN_SOCK[PENJUALAN_SOCK.jenis == "retur"].groupby("kode_barang")["qty"].sum()
ms = pd.DataFrame(index=SOCKS.index)
ms["divisi"] = "Kaos Kaki (acuan Flipkart)"; ms["kategori"] = "Kaos Kaki"; ms["subkategori"] = SOCKS["nama_barang"]; ms["supplier"] = SOCKS["brand"]; ms["nama_segmen"] = SOCKS["product_name"]
ms["harga_jual_ref"] = SOCKS["harga_jual_idr"] / HARGA_SKALA; ms["harga_beli_ref"] = SOCKS["harga_beli_idr"] / HARGA_SKALA    # dikali HARGA_SKALA kembali di bawah -> tepat rupiah
ms["harga_beli_diimputasi"] = True; ms["nama_barang"] = SOCKS["nama_barang"]; ms["diskon_persen"] = SOCKS["diskon_persen"]
ms["unit_keluar"] = _k.reindex(ms.index).fillna(0).astype(int); ms["unit_retur"] = _r.reindex(ms.index).fillna(0).astype(int)
ms["minimum_stok"] = np.clip(np.ceil(ms["unit_keluar"] / (N_HARI / 7) * LEAD_WEEKS), 3, 50).astype(int)
for kode in ms.index:
    lam = _lam[kode]; rendah = rng.random() < 0.10
    for g, share in zip(GUDANG_IDS, GUDANG_SHARE):
        F_ = int(rng.integers(0, max(2, ms.loc[kode, "minimum_stok"]))) if rendah else int(rng.lognormal(np.log(max(lam * share * 45, 1.0)), 0.5))
        STOK_F[(kode, g)] = F_; CEK_G[(kode, g)] = TGL_MAX - timedelta(days=int(rng.integers(0, 120)))
ms["stok_total_bersih"] = [sum(STOK_F[(k, g)] for g in GUDANG_IDS) for k in ms.index]; ms["stok_total_mentah"] = ms["stok_total_bersih"]
ms["n_toko_aktif"] = len(GUDANG_IDS); ms["punya_stok_aktif"] = True; ms["sumber_data"] = "flipkart_produk_asli+stok_penjualan_sintetis"
MASTER = pd.concat([MASTER, ms]); MASTER.index.name = "kode_barang"
print(f"Kaos kaki Flipkart: {len(SOCKS)} produk | penjualan sintetis {int(ms['unit_keluar'].sum())} unit | stok total sintetis {int(ms['stok_total_bersih'].sum())}")
N_RETAIL = N_SKU_APP - len(SOCKS)

# ---- pilih ~N_RETAIL produk: bertingkat per kategori, peluang ~ akar volume penjualan ----
elig = MASTER[MASTER["sumber_data"].str.startswith("retail") & (MASTER["stok_total_bersih"] > 0) & (MASTER["unit_keluar"] >= MIN_UNIT_TERJUAL)]
cnt = elig.groupby("kategori").size()
alok = np.minimum(cnt, np.maximum(2, np.floor(cnt / cnt.sum() * N_RETAIL).astype(int)))
while alok.sum() < N_RETAIL:
    sisa = (cnt - alok); sisa = sisa[sisa > 0]
    if sisa.empty: break
    alok[(sisa / cnt[sisa.index]).idxmax()] += 1
while alok.sum() > N_RETAIL:
    alok[alok.idxmax()] -= 1
SKU = []
for kat, k in alok.items():
    pool = elig[elig["kategori"] == kat]; w = np.sqrt(pool["unit_keluar"].to_numpy(float)); w = w / w.sum()
    SKU += list(rng.choice(pool.index.to_numpy(), size=int(k), replace=False, p=w))
SKU = sorted(SKU + list(SOCKS.index)); APP = MASTER.loc[SKU].copy()
print(f"Produk aplikasi: {len(APP)} ({N_RETAIL} retail dari {len(elig)} kandidat + {len(SOCKS)} kaos kaki) | kategori: {APP['kategori'].nunique()} | divisi: {APP['divisi'].value_counts().to_dict()}")

# ---- pasangkan acuan Flipkart (gambar, keterangan, berat) per divisi ----
ACUAN = {}
for div, grp in APP.groupby("divisi"):
    if div not in POOL: continue
    pool = POOL[div]; idx = rng.choice(len(pool), size=len(grp), replace=len(pool) < len(grp))
    for kode, i in zip(grp.index, idx): ACUAN[kode] = pool.iloc[int(i)]
for k in SOCKS.index: ACUAN[k] = SOCKS.loc[k]
DF_ACUAN = pd.DataFrame([dict(kode_barang=k, flipkart_pid=r["pid"], flipkart_nama=r["product_name"], kelompok=APP.loc[k, "divisi"],
                              metode="produk asli Flipkart" if k.startswith("FKS-") else "acak berbasis kategori (bukan padanan produk asli)") for k, r in ACUAN.items()])
APP["gambar"] = [ACUAN[k]["gambar"] for k in APP.index]
APP["keterangan"] = ["Data contoh. " + (ACUAN[k]["ket"] or APP.loc[k, "subkategori"])[:150] for k in APP.index]
APP["berat_gram"] = [ACUAN[k]["berat_gram"] for k in APP.index]
APP["satuan"] = np.where(APP["divisi"].str.contains("Footwear|Kaos Kaki"), "pasang", "pcs")
APP["harga_beli"] = (APP["harga_beli_ref"] * HARGA_SKALA).round(0)
APP["harga_jual"] = (APP["harga_jual_ref"] * HARGA_SKALA).round(0)
assert (APP["harga_beli"] < APP["harga_jual"]).all()

# kategori & rak
KAT_LIST = sorted(APP["kategori"].unique()); KAT_ID = {k: i for i, k in enumerate(KAT_LIST, start=1)}
APP["id_kategori"] = APP["kategori"].map(KAT_ID)
RAK = {}
for kode in SKU:
    for g in GUDANG_IDS: RAK[(kode, g)] = f"{chr(64 + g)}{KAT_ID[APP.loc[kode, 'kategori']]:02d}-{int(rng.integers(1, 6))}"
print("Harga contoh:", APP[["nama_barang", "harga_beli", "harga_jual", "diskon_persen", "minimum_stok"]].head(3).to_string(index=False))

Beban stok per gudang: {'Gudang A': 12909, 'Gudang B': 12323, 'Gudang C': 12344}
Kaos kaki Flipkart: 33 produk | penjualan sintetis 5950 unit | stok total sintetis 904
Produk aplikasi: 300 (267 retail dari 1810 kandidat + 33 kaos kaki) | kategori: 28 | divisi: {'Scholar Footwear': 96, 'Junior Apparel': 87, 'Femme Footwear': 84, 'Kaos Kaki (acuan Flipkart)': 33}
Harga contoh:                                                          nama_barang  harga_beli  harga_jual  diskon_persen  minimum_stok
                                NBA Women's Solid Ankle Length Socks     42500.0     85500.0           40.0             6
Bonjour Ladies Socks Women's Self Design, Striped Ankle Length Socks     36000.0     72000.0            0.0             3
 Pumpkin Patch Baby Girl's, Baby Boy's Self Design Crew Length Socks     33000.0     66500.0            0.0             4


## 6. Tabel database dan riwayat (commit)
Penjualan menjadi barang keluar, retur menjadi barang masuk. Karena data sumber tidak memuat penerimaan barang, saldo awal, penerimaan, dan penyesuaian akhir **diestimasi** agar seluruh riwayat berujung tepat di snapshot stok.

In [6]:
# ============ 6. TABEL DATABASE: master, stok, dan riwayat (commit) dari data penjualan ============
T = defaultdict(list); IDS = defaultdict(int)
def nid(t): IDS[t] += 1; return IDS[t]

for g, nm in zip(GUDANG_IDS, GUDANG_NAMA): T["gudang"].append(dict(id_gudang=g, nama_gudang=nm))
for k, i in KAT_ID.items():
    T["kategori_barang"].append(dict(id_kategori=i, nama_kategori=k, deskripsi=(f"Kategori kaos kaki dari produk Flipkart ({k})" if "Kaos Kaki" in k else f"Kategori dari dataset retail ({k})"), divisi=APP[APP["kategori"] == k]["divisi"].iloc[0]))

HASH = (bcrypt.hashpw(DEMO_PASSWORD.encode(), bcrypt.gensalt(rounds=10)).decode() if bcrypt else "$2b$10$GANTI_DENGAN_HASH_DEMO")
_u = [("Dewi Lestari", "owner@contoh.test", "owner", "aktif", None), ("Rendra Pratama", "spv1@contoh.test", "spv", "aktif", "Pagi"),
      ("Sari Anggraini", "spv2@contoh.test", "spv", "aktif", "Siang"), ("Bagas Wibowo", "staff1@contoh.test", "staff", "aktif", "Pagi"),
      ("Putri Maharani", "staff2@contoh.test", "staff", "aktif", "Siang"), ("Dimas Saputra", "staff3@contoh.test", "staff", "aktif", "Pagi"),
      ("Nadia Rahma", "staff4@contoh.test", "staff", "belum_aktif", "Siang")]
for i, (nm, em, role, st, shift) in enumerate(_u, start=1):
    T["pengguna"].append(dict(id_pengguna=i, nama_lengkap=nm, email=em, password_hash=HASH if st == "aktif" else None,
        no_telepon="08" + "".join(str(x) for x in rng.integers(0, 10, 10)), alamat="Samarinda (alamat contoh)", status_akun=st, role=role,
        wajib_ganti_password=(st == "aktif" and role != "owner"), shift_kerja=shift,
        nama_usaha="Toko Contoh (data contoh)" if role == "owner" else None, alamat_usaha="Samarinda (alamat contoh)" if role == "owner" else None,
        created_at=at(TGL_MIN - timedelta(days=30), 9), updated_at=None))
SPV_IDS = [u["id_pengguna"] for u in T["pengguna"] if u["role"] == "spv"]
STAFF_IDS = [u["id_pengguna"] for u in T["pengguna"] if u["role"] == "staff" and u["status_akun"] == "aktif"]
T["token_aktivasi"].append(dict(id_token=1, id_pengguna=7, token_hash="b" * 64, kedaluwarsa_pada=at(TGL_MAX + timedelta(days=7), 23, 59), dipakai_pada=None, created_at=at(TGL_MAX, 10)))

BARANG = []
for kode, r in APP.iterrows():
    BARANG.append(dict(kode_barang=kode, nama_barang=r["nama_barang"], id_kategori=int(r["id_kategori"]), harga_beli=float(r["harga_beli"]), harga_jual=float(r["harga_jual"]),
        minimum_stok=int(r["minimum_stok"]), tipe_diskon=1 if r["diskon_persen"] > 0 else 0, diskon=float(r["diskon_persen"]), satuan=r["satuan"],
        berat_gram=None if pd.isna(r["berat_gram"]) else int(r["berat_gram"]), keterangan=r["keterangan"][:255], gambar=r["gambar"],
        sumber_data=r["sumber_data"], created_at=at(TGL_MIN - timedelta(days=30), 9), updated_at=None))
T["barang"] = BARANG
BY_KODE = {b["kode_barang"]: b for b in BARANG}

# ---- peristiwa stok per (barang, gudang): saldo awal -> penjualan/retur (+ penerimaan diestimasi) -> penyesuaian akhir ----
pj = PENJUALAN[PENJUALAN["kode_barang"].isin(SKU)].copy(); pj["id_gudang"] = pj["toko"].map(TOKO_GUDANG)
pj = pj.groupby(["tanggal", "kode_barang", "id_gudang", "jenis"], as_index=False)["qty"].sum()
by_pair = defaultdict(list)
for r in pj.itertuples(index=False):
    by_pair[(r.kode_barang, r.id_gudang)].append((r.tanggal.date(), 2 if r.jenis == "keluar" else 3, -int(r.qty) if r.jenis == "keluar" else int(r.qty)))
EV = []; n_restock = n_adj = 0
for kode in SKU:
    for g in GUDANG_IDS:
        evs = sorted(by_pair.get((kode, g), [])); F = STOK_F.get((kode, g), 0)
        ru = max(5, math.ceil(sum(-d for _, _, d in evs if d < 0) / N_HARI * 30))
        stock, seq = 0, []
        for dt, o, d in evs:
            if d < 0 and stock + d < 0:
                r_ = -(stock + d) + ru; seq.append((dt, 1, r_)); stock += r_; n_restock += 1
            seq.append((dt, o, d)); stock += d
        delta = F - stock
        if delta > 0: EV.append((TGL_MIN, 0, kode, g, delta))
        EV += [(dt, o, kode, g, d) for dt, o, d in seq]
        if delta < 0: EV.append((TGL_MAX, 4, kode, g, delta)); n_adj += 1
GT = {}
def waktu_grup(dt, g, grp):
    k = (dt, g, grp)
    if k not in GT:
        GT[k] = {0: lambda: at(dt, 7, 30 + g), 1: lambda: at(dt, 8, 30 + g), 2: lambda: at(dt, int(rng.integers(11, 14)), int(rng.integers(0, 60))),
                 3: lambda: at(dt, 15, g), 4: lambda: at(dt, 16, 45 + g)}[grp]()
    return GT[k]
EV.sort(key=lambda e: (e[0], e[1], e[3], e[2]))
ALASAN = {0: "Saldo awal pencatatan sistem (diturunkan dari snapshot inventory)", 1: "Penerimaan barang (diestimasi dari pola penjualan, bukan data nyata)",
          2: "Penjualan (dari data penjualan retail)", 3: "Retur pelanggan (dari data penjualan retail)", 4: "Penyesuaian selisih stok akhir terhadap snapshot inventory"}
JENIS_TRX = {0: "masuk", 1: "masuk", 2: "keluar", 3: "masuk", 4: "penyesuaian"}
SIS = defaultdict(int); ITEM = {}; TRX = {}
batas_pending = TGL_MAX - timedelta(days=HARI_PENDING)
for dt, grp, kode, g, d in sorted(EV, key=lambda e: (waktu_grup(e[0], e[3], e[1]), e[3], e[1], e[2])):
    w = waktu_grup(dt, g, grp); gk = (dt, g, grp)
    if gk not in TRX:
        user = SPV_IDS[0] if grp in (0, 4) else int(rng.choice(STAFF_IDS)); id_t = nid("transaksi_stok"); TRX[gk] = (id_t, user)
        T["transaksi_stok"].append(dict(id_transaksi=id_t, id_pengguna=user, jenis_transaksi=JENIS_TRX[grp], waktu_transaksi=w, status_transaksi="Selesai", alasan=ALASAN[grp], alasan_pembatalan=None))
    id_t, user = TRX[gk]
    T["detail_transaksi_stok"].append(dict(id_detail_transaksi=nid("detail_transaksi_stok"), id_transaksi=id_t, kode_barang=kode,
        id_gudang_asal=g if grp == 2 else None, id_gudang_tujuan=None if grp == 2 else g, jumlah=int(d) if grp == 4 else abs(int(d)),
        stok_target=int(STOK_F.get((kode, g), 0)) if grp == 4 else None, no_batch=None, catatan=None))
    before = SIS[(kode, g)]; after = before + int(d); assert after >= 0, (kode, g, before, d); SIS[(kode, g)] = after
    rid = nid("riwayat_aktivitas")
    T["riwayat_aktivitas"].append(dict(id_riwayat=rid, id_pengguna=user, id_transaksi=id_t, id_detail_opname=None, kode_barang=kode, id_gudang=g,
        jenis_aktivitas=JENIS_TRX[grp], waktu_aktivitas=w, jumlah_sebelum=before, jumlah_sesudah=after, jumlah_perubahan=int(d), alasan=ALASAN[grp], id_riwayat_ref=None))
    it = nid("item_laporan"); pending = dt >= batas_pending
    row = dict(id_item_laporan=it, id_riwayat=rid, status_item="Diajukan" if pending else "Diterima", diperiksa_oleh=None if pending else SPV_IDS[0], diajukan_pada=w)
    T["item_laporan"].append(row); ITEM[it] = row
    if not pending:
        spv = SPV_IDS[0] if grp == 0 else int(rng.choice(SPV_IDS))
        wv = at(dt, 9, 15) if grp == 0 else at(dt + timedelta(days=1), int(rng.integers(9, 16)), int(rng.integers(0, 60)))
        row["diperiksa_oleh"] = spv
        T["riwayat_validasi"].append(dict(id_riwayat_validasi=nid("riwayat_validasi"), id_item_laporan=it, id_spv=spv, status_sebelumnya="Diajukan", status_baru="Diterima", tanggal_validasi=wv, catatan_validasi=None))

for kode in SKU:
    for g in GUDANG_IDS:
        T["stok_barang"].append(dict(id_stok=nid("stok_barang"), kode_barang=kode, id_gudang=g, jumlah_stok=SIS[(kode, g)], letak_rak=RAK[(kode, g)], tanggal_pengecekan_terakhir=CEK_G.get((kode, g))))
TOTAL = {k: sum(SIS[(k, g)] for g in GUDANG_IDS) for k in SKU}
print({t: len(v) for t, v in T.items()})
print(f"Penerimaan barang diestimasi: {n_restock} | penyesuaian akhir (selisih penjualan vs snapshot): {n_adj} dari {len(SKU)*len(GUDANG_IDS)} pasangan")

{'gudang': 3, 'kategori_barang': 28, 'pengguna': 7, 'token_aktivasi': 1, 'barang': 300, 'transaksi_stok': 2585, 'detail_transaksi_stok': 23561, 'riwayat_aktivitas': 23561, 'item_laporan': 23561, 'riwayat_validasi': 23083, 'stok_barang': 900}
Penerimaan barang diestimasi: 3251 | penyesuaian akhir (selisih penjualan vs snapshot): 320 dari 900 pasangan


## 7. Berkas Kasir Pintar dan skenario uji
Berkas ekspor memakai stok **mentah** (termasuk negatif), sehingga selisih dengan stok website muncul alami dari data kotor.

In [7]:
# ============ 7. BERKAS KASIR PINTAR (kolom TEMPLATE_BARANG) DAN SKENARIO UJI ============
TEMPLATE_COLS = ["alasan_gagal", "data_kode_barang", "data_nama_barang", "data_harga_beli", "data_harga_jual", "data_stok", "data_barang_jasa", "data_show_toko",
                 "minimum_stok", "tipe_diskon", "diskon", "berat_dan_satuan", "berat", "letak_rak", "keterangan", "kategori", "gambar"]
def baris_template(kode, stok, nama=None, harga_beli=None, harga_jual=None, kategori=None, minimum=None, diskon=0.0, berat=None, rak="", ket="Data contoh", gambar=""):
    return {"alasan_gagal": "", "data_kode_barang": kode, "data_nama_barang": nama, "data_harga_beli": harga_beli, "data_harga_jual": harga_jual, "data_stok": stok,
            "data_barang_jasa": 0, "data_show_toko": 0, "minimum_stok": minimum, "tipe_diskon": 1 if diskon > 0 else 0, "diskon": diskon,
            "berat_dan_satuan": "gram" if berat else "", "berat": berat if berat else "", "letak_rak": rak, "keterangan": ket, "kategori": kategori, "gambar": gambar}
def dari_app(kode, stok):
    b = BY_KODE[kode]
    return baris_template(kode, stok, b["nama_barang"], b["harga_beli"], b["harga_jual"], KAT_LIST[b["id_kategori"] - 1], b["minimum_stok"], b["diskon"],
                          b["berat_gram"], RAK[(kode, GUDANG_IDS[0])], b["keterangan"], b["gambar"])

# (a) master barang aplikasi dalam format template (stok = jumlah semua gudang)
df_template = pd.DataFrame([dari_app(k, TOTAL[k]) for k in SKU], columns=TEMPLATE_COLS)
df_template.to_excel(f"{OUT_DIR}/fs/kasir_pintar/barang_aplikasi_format_template.xlsx", index=False)

# (b) berkas ekspor Kasir Pintar: stok MENTAH dari inventory (termasuk stok negatif) -> selisih muncul alami dari data kotor
luar = MASTER[MASTER["punya_stok_aktif"] & ~MASTER.index.isin(SKU)]
luar_pilih = list(rng.choice(luar.index.to_numpy(), size=5, replace=False))
rows = [dari_app(k, int(MASTER.loc[k, "stok_total_mentah"])) for k in SKU]
for k in luar_pilih:
    m = MASTER.loc[k]; rows.append(baris_template(k, int(m["stok_total_mentah"]), m["nama_barang"], round(m["harga_beli_ref"] * HARGA_SKALA), round(m["harga_jual_ref"] * HARGA_SKALA),
                                                  m["kategori"], int(m["minimum_stok"]), 0.0, None, "", "Data contoh (produk di luar 300 produk aplikasi)"))
df_kasir = pd.DataFrame(rows, columns=TEMPLATE_COLS)
df_kasir.to_excel(f"{OUT_DIR}/fs/kasir_pintar/kasir_benar.xlsx", index=False); df_kasir.to_csv(f"{OUT_DIR}/fs/kasir_pintar/kasir_benar.csv", index=False)
exp = []
for k, q in zip(df_kasir["data_kode_barang"], df_kasir["data_stok"]):
    if k not in BY_KODE: exp.append(dict(kode_barang=k, jumlah_berkas=q, stok_website_total=None, selisih_berkas_minus_website=None, status_diharapkan="tidak_ditemukan"))
    else: exp.append(dict(kode_barang=k, jumlah_berkas=q, stok_website_total=TOTAL[k], selisih_berkas_minus_website=q - TOTAL[k], status_diharapkan="sesuai" if q == TOTAL[k] else "beda_jumlah"))
df_exp = pd.DataFrame(exp); df_exp.to_csv(f"{OUT_DIR}/fs/kasir_pintar/hasil_diharapkan_pencocokan.csv", index=False)

# (c) uji negatif
df_kasir.head(10).to_csv(f"{OUT_DIR}/fs/kasir_pintar/kasir_format_salah.txt", index=False, sep="\t")
df_kasir.drop(columns=["data_kode_barang"]).head(10).to_excel(f"{OUT_DIR}/fs/kasir_pintar/kasir_tanpa_kolom_kode.xlsx", index=False)
pd.DataFrame([baris_template(f"XXX-{i:03d}", 5, f"Barang Tak Dikenal {i}", 10000, 20000, "Lainnya") for i in range(1, 6)], columns=TEMPLATE_COLS).to_excel(
    f"{OUT_DIR}/fs/kasir_pintar/kasir_semua_kode_tidak_ada.xlsx", index=False)
pd.DataFrame(columns=TEMPLATE_COLS).to_excel(f"{OUT_DIR}/fs/kasir_pintar/kasir_kosong.xlsx", index=False)

# (d) skenario wajib plan: 5 tercatat, 3 tersedia fisik -> selisih 2 barang
cand = [k for k in SKU if TOTAL[k] >= 5]; pick = [str(x) for x in rng.choice(cand, size=5, replace=False)]
fis = {k: (TOTAL[k] if i < 3 else TOTAL[k] - 2) for i, k in enumerate(pick)}
pd.DataFrame([dari_app(k, TOTAL[k]) for k in pick], columns=TEMPLATE_COLS).to_excel(f"{OUT_DIR}/fs/skenario_uji/kasir_skenario_5_tercatat.xlsx", index=False)
pd.DataFrame([dict(kode_barang=k, nama_barang=BY_KODE[k]["nama_barang"], tercatat=TOTAL[k], fisik=fis[k], selisih_diharapkan=TOTAL[k] - fis[k],
                   status_awal_diharapkan="Belum dicek" if TOTAL[k] - fis[k] else "sesuai") for k in pick]).to_csv(f"{OUT_DIR}/fs/skenario_uji/skenario_5_3_hitung_fisik.csv", index=False)
neg = int((df_kasir["data_stok"] < 0).sum())
print("Kasir benar:", len(df_kasir), "baris | kunci jawaban:", df_exp["status_diharapkan"].value_counts().to_dict(), "| stok negatif di berkas:", neg)

Kasir benar: 305 baris | kunci jawaban: {'sesuai': 266, 'beda_jumlah': 34, 'tidak_ditemukan': 5} | stok negatif di berkas: 6


## 8. Dataset intent untuk tim AI
Lima kelas. Entitas (kategori, barang, kode, gudang) diambil dari data produk. Data uji dipisah per template untuk mencegah kebocoran.

In [8]:
# ============ 8. DATASET INTENT UNTUK TIM AI (kalimat berlabel, entitas dari data nyata) ============
BARANG_ENT = APP["subkategori"].drop_duplicates().tolist()
KATEGORI_ENT = KAT_LIST
KODE_ENT = SKU
GUDANG_ENT = [n for n in GUDANG_NAMA] + [n.lower() for n in GUDANG_NAMA] + [n.replace("Gudang", "gdg") for n in GUDANG_NAMA]

TEMPLATES = {
 "cek_stok": [
  "stok {barang} di {gudang} berapa", "berapa sisa stok {barang}", "cek stok {kode}", "{barang} di {gudang} masih ada gak", "tolong cek jumlah stok {barang} di {gudang}",
  "stok kategori {kategori} tersisa berapa", "barang apa saja yang stoknya menipis di {gudang}", "ada berapa unit {barang} di {gudang}", "kode {kode} stoknya tinggal berapa",
  "total stok {barang} di semua gudang", "info stok {kode} di {gudang}", "apakah {barang} masih tersedia", "stok {kode} {gudang} ada berapa", "sisa barang {kategori} di {gudang}",
  "jumlah stok {kode} sekarang", "barang yang stoknya habis di {gudang} apa saja", "masih ada stok {barang} gak", "lihat stok {barang} per gudang", "stok {kode} di semua gudang",
  "berapa banyak {kategori} yang ada di {gudang}", "cek ketersediaan {kode}", "barang mana yang stoknya di bawah minimum"],
 "status_selisih": [
  "selisih barang {kode} sudah ditindaklanjuti belum", "bagaimana status selisih {barang}", "ada berapa selisih yang belum dicek", "selisih {kode} statusnya apa sekarang",
  "daftar selisih yang menunggu validasi", "selisih mana yang masih sedang dicek ulang", "apakah selisih {barang} sudah selesai", "berapa jumlah selisih hasil impor kasir pintar terakhir",
  "selisih stok {barang} di {gudang} berapa", "barang apa saja yang stoknya tidak cocok dengan kasir pintar", "selisih {kode} sudah divalidasi spv belum", "tampilkan semua selisih yang belum selesai",
  "ada selisih baru di {gudang} gak", "status tindak lanjut selisih {kode}", "berapa barang yang selisihnya belum dicek ulang", "selisih {kategori} yang masih terbuka apa saja",
  "hasil pencocokan kasir pintar untuk {kode} bagaimana", "selisih di {gudang} sudah beres belum", "ada barang yang jumlahnya beda dengan kasir pintar", "cek status selisih {barang}",
  "selisih mana yang perlu saya validasi", "jumlah selisih yang sudah selesai ada berapa"],
 "riwayat": [
  "siapa yang terakhir mengubah stok {barang}", "tampilkan riwayat {barang} di {gudang}", "riwayat aktivitas {kode} minggu ini", "kapan terakhir {barang} keluar dari {gudang}",
  "catatan perpindahan {barang} antar gudang", "aktivitas staff hari ini apa saja", "lihat riwayat barang masuk di {gudang}", "siapa yang mencatat barang keluar {kode}",
  "riwayat penyesuaian stok {barang}", "kapan {kode} terakhir dicek", "history stok {kode}", "tampilkan aktivitas terakhir di {gudang}", "siapa yang memindahkan {barang} ke {gudang}",
  "riwayat retur {barang}", "lihat perubahan stok {kode} bulan lalu", "aktivitas yang ditolak spv minggu ini apa saja", "kapan terakhir ada barang masuk di {gudang}",
  "riwayat {kategori} di {gudang}", "siapa yang menyetujui aktivitas {kode}", "tampilkan commit terakhir untuk {barang}", "jejak perubahan stok {kode}", "catatan penyesuaian di {gudang}"],
 "prosedur_stok": [
  "bagaimana cara mencatat barang masuk", "cara mencatat barang keluar bagaimana", "langkah memindahkan barang antar gudang", "bagaimana cara input stok fisik", "apa itu penyesuaian stok",
  "kapan saya harus membuat penyesuaian stok", "bagaimana cara mengunggah berkas kasir pintar", "format berkas kasir pintar yang benar seperti apa", "apa yang harus dilakukan kalau aktivitas saya ditolak spv",
  "bagaimana cara menindaklanjuti selisih", "siapa yang boleh menerima atau menolak aktivitas staff", "apa arti status diajukan diterima dan ditolak", "bagaimana cara menghitung ulang stok fisik di gudang",
  "kenapa barang keluar tidak bisa melebihi stok", "bagaimana alur menangani selisih barang", "apa bedanya stok sistem dan stok fisik", "cara mengaktifkan akun lewat email", "bagaimana cara mengubah password",
  "apakah riwayat aktivitas bisa dihapus", "bagaimana cara mengisi alasan penyesuaian", "kapan stok dianggap menipis", "apa yang dimaksud minimum stok", "bagaimana prosedur opname stok", "bagaimana cara melihat riwayat barang"],
}
OOC_BASE = [
 "cuaca hari ini bagaimana", "siapa presiden indonesia sekarang", "tolong buatkan puisi tentang hujan", "berapa kurs dolar hari ini", "resep nasi goreng yang enak", "jadwal bioskop malam ini",
 "apa ibu kota prancis", "ceritakan lelucon lucu", "cara membuat kue bolu", "harga bitcoin sekarang berapa", "berapa omzet penjualan bulan ini", "siapa pemenang piala dunia terakhir",
 "bagaimana cara membuat website", "rekomendasi film terbaru", "terjemahkan kalimat ini ke bahasa inggris", "cara menurunkan berat badan", "berapa harga sepatu di toko sebelah",
 "bagaimana cara berjualan lewat whatsapp", "cara mengemas dan mengirim paket", "berapa total keuntungan minggu ini", "tolong hitung 125 dikali 8", "apa kabar hari ini", "kamu siapa",
 "bagaimana cara deep cleaning gudang", "rekomendasi tempat makan di samarinda", "cara memperbaiki laptop yang lambat", "siapa penemu telepon", "buatkan caption instagram untuk produk",
 "berapa gaji karyawan toko", "laporan keuangan bulan lalu", "tips diet sehat", "bagaimana cara membuka cabang baru", "kapan tahun baru imlek", "ajarkan saya bahasa jepang",
 "apa itu machine learning", "berapa jumlah penduduk indonesia", "lagu populer minggu ini", "cara menanam cabai di pot", "cuaca besok di jakarta", "tolong ringkas berita hari ini"]
OOC_UJI = ["apa warna favorit kamu", "berapa tinggi gunung everest", "cara membuat teh tarik", "siapa yang menang pertandingan semalam", "bagaimana cara investasi saham",
           "tolong tuliskan surat lamaran kerja", "harga emas hari ini", "kenapa langit berwarna biru", "bagaimana cara memasak rendang", "rekomendasi game seru", "berapa penjualan kemarin",
           "cara kirim barang ke luar negeri", "jam buka bank sekarang", "bantu saya belajar python", "apa arti kata nostalgia"]
assert not set(OOC_UJI) & set(OOC_BASE)
AWAL = ["", "", "", "tolong ", "min ", "kak ", "bisa tolong ", "permisi, ", "halo, ", "coba "]
AKHIR = ["", "", "", " ya", " dong", "?", " sekarang", " please", " nih", " tolong"]
def typo(t):
    w = t.split(); idx = [i for i, x in enumerate(w) if len(x) >= 5 and x.isalpha()]
    if not idx: return t
    i = int(rng.choice(idx)); x = w[i]; j = int(rng.integers(1, len(x) - 2)); w[i] = x[:j] + x[j + 1] + x[j] + x[j + 2:]
    return " ".join(w)
def norm(t): return re.sub(r"[^a-z0-9 ]", "", t.lower()).strip()
def isi(tpl):
    return tpl.format(barang=str(rng.choice(BARANG_ENT)), kategori=str(rng.choice(KATEGORI_ENT)), kode=str(rng.choice(KODE_ENT)), gudang=str(rng.choice(GUDANG_ENT)))

# pembagian uji berbasis TEMPLATE (bukan acak per kalimat) agar kalimat mirip tidak bocor ke data uji
TEMPLATES["di_luar_konteks"] = OOC_BASE
BARIS = []
for intent, tpls in TEMPLATES.items():
    ids = list(range(len(tpls))); n_uji = max(2, round(0.2 * len(ids)))
    uji_ids = set(rng.choice(ids, size=n_uji, replace=False).tolist())
    seen = set(); tries = 0; kuota = {"latih": round(TARGET_PER_INTENT * 0.8), "uji": TARGET_PER_INTENT - round(TARGET_PER_INTENT * 0.8)}; hitung = {"latih": 0, "uji": 0}
    while (hitung["latih"] < kuota["latih"] or hitung["uji"] < kuota["uji"]) and tries < 20000:
        tries += 1; ti = int(rng.integers(0, len(tpls))); split = "uji" if ti in uji_ids else "latih"
        if hitung[split] >= kuota[split]: continue
        teks = (str(rng.choice(AWAL)) + isi(tpls[ti]) + str(rng.choice(AKHIR))).strip()
        if rng.random() < 0.12: teks = typo(teks)
        if norm(teks) in seen: continue
        seen.add(norm(teks)); hitung[split] += 1
        BARIS.append(dict(teks=teks, intent=intent, template_id=f"{intent}_{ti:02d}", split=split, sumber="template_tim_entitas_dari_data_retail"))
DF_INTENT = pd.DataFrame(BARIS).sample(frac=1, random_state=SEED).reset_index(drop=True)
DF_INTENT.to_csv(f"{OUT_DIR}/ai/dataset_intent_lengkap.csv", index=False)
DF_INTENT[DF_INTENT.split == "latih"].to_csv(f"{OUT_DIR}/ai/dataset_intent_latih.csv", index=False)
DF_INTENT[DF_INTENT.split == "uji"].to_csv(f"{OUT_DIR}/ai/dataset_intent_uji.csv", index=False)
pd.DataFrame({"teks": OOC_UJI, "intent": "di_luar_konteks", "keterangan": "uji penolakan sopan; TIDAK ada di data latih"}).to_csv(f"{OUT_DIR}/ai/uji_di_luar_konteks_holdout.csv", index=False)
pd.DataFrame([
 dict(intent="cek_stok", deskripsi="Menanyakan jumlah/ketersediaan stok barang atau kategori", sumber_jawaban="stok_barang, barang, v_stok_ringkas", contoh=TEMPLATES["cek_stok"][0]),
 dict(intent="status_selisih", deskripsi="Menanyakan selisih dan status tindak lanjutnya", sumber_jawaban="selisih_stok, tindak_lanjut_selisih, perbandingan_stok", contoh=TEMPLATES["status_selisih"][0]),
 dict(intent="riwayat", deskripsi="Menanyakan riwayat aktivitas (commit) barang/gudang/pengguna", sumber_jawaban="riwayat_aktivitas, item_laporan", contoh=TEMPLATES["riwayat"][0]),
 dict(intent="prosedur_stok", deskripsi="Menanyakan cara/prosedur memakai aplikasi (dijawab lewat RAG)", sumber_jawaban="korpus panduan buatan tim (belum dibuat)", contoh=TEMPLATES["prosedur_stok"][0]),
 dict(intent="di_luar_konteks", deskripsi="Pertanyaan di luar stok; ditolak sopan, Gemini tidak dipanggil", sumber_jawaban="-", contoh=OOC_BASE[0])]).to_csv(f"{OUT_DIR}/ai/kamus_intent.csv", index=False)
print(DF_INTENT.groupby(["intent", "split"]).size().unstack().to_string())
print("Contoh:", DF_INTENT.sample(4, random_state=1)[["teks", "intent"]].to_dict("records"))

split            latih  uji
intent                     
cek_stok            80   20
di_luar_konteks     80   20
prosedur_stok       80   20
riwayat             80   20
status_selisih      80   20
Contoh: [{'teks': 'min hasil pencocokan kasir pintar untuk PROD-145911 bagaimana', 'intent': 'status_selisih'}, {'teks': "min ada berapa unit Texlon Men's Mid-calf Length Socks di gdg A", 'intent': 'cek_stok'}, {'teks': 'min tolong cek jumlah stok Junior Apparel Crew Tees Zephyr di gudang b ya', 'intent': 'cek_stok'}, {'teks': 'jumlah stok PROD-103738 sekarang', 'intent': 'cek_stok'}]


## 9. Validasi otomatis
Termasuk keterlacakan ke data mentah: total keluar dan retur retail di riwayat harus sama dengan data sumber, dan stok akhir harus sama dengan snapshot inventory bersih. Produk kaos kaki diperiksa berasal dari Flipkart dan berlabel sintetis.

In [9]:
# ============ 9. VALIDASI OTOMATIS ============
COLS = {
 "gudang": ["id_gudang","nama_gudang"],
 "kategori_barang": ["id_kategori","nama_kategori","deskripsi","divisi"],
 "pengguna": ["id_pengguna","nama_lengkap","email","password_hash","no_telepon","alamat","status_akun","role","wajib_ganti_password","shift_kerja","nama_usaha","alamat_usaha","created_at","updated_at"],
 "token_aktivasi": ["id_token","id_pengguna","token_hash","kedaluwarsa_pada","dipakai_pada","created_at"],
 "barang": ["kode_barang","nama_barang","id_kategori","harga_beli","harga_jual","minimum_stok","tipe_diskon","diskon","satuan","berat_gram","keterangan","gambar","sumber_data","created_at","updated_at"],
 "stok_barang": ["id_stok","kode_barang","id_gudang","jumlah_stok","letak_rak","tanggal_pengecekan_terakhir"],
 "transaksi_stok": ["id_transaksi","id_pengguna","jenis_transaksi","waktu_transaksi","status_transaksi","alasan","alasan_pembatalan"],
 "detail_transaksi_stok": ["id_detail_transaksi","id_transaksi","kode_barang","id_gudang_asal","id_gudang_tujuan","jumlah","stok_target","no_batch","catatan"],
 "stock_opname": ["id_opname","id_gudang","id_pengguna","waktu_opname","status_opname","catatan"],
 "detail_stok_opname": ["id_detail_opname","id_opname","kode_barang","stok_sistem","stok_fisik","jumlah_selisih","keterangan"],
 "riwayat_aktivitas": ["id_riwayat","id_pengguna","id_transaksi","id_detail_opname","kode_barang","id_gudang","jenis_aktivitas","waktu_aktivitas","jumlah_sebelum","jumlah_sesudah","jumlah_perubahan","alasan","id_riwayat_ref"],
 "item_laporan": ["id_item_laporan","id_riwayat","status_item","diperiksa_oleh","diajukan_pada"],
 "riwayat_validasi": ["id_riwayat_validasi","id_item_laporan","id_spv","status_sebelumnya","status_baru","tanggal_validasi","catatan_validasi"],
 "import_kasir": ["id_import","id_spv","nama_file","format_file","tanggal_import","keterangan"],
 "detail_import": ["id_detail_import","id_import","nomor_baris","kode_barang_berkas","nama_barang_berkas","jumlah_berkas","status_pencocokan","alasan_gagal"],
 "perbandingan_stok": ["id_perbandingan_stok","id_detail_import","kode_barang","jenis_sumber","stok_website","stok_pembanding","selisih","status_perbandingan","tanggal_perbandingan"],
 "selisih_stok": ["id_selisih_stok","id_perbandingan_stok","id_detail_opname","jenis_selisih","status_kasus","keterangan","created_at"],
 "tindak_lanjut_selisih": ["id_tindak_lanjut","id_selisih_stok","id_pengguna","jenis_tindakan","tanggal_tindak_lanjut","catatan"],
 "notifikasi": ["id_notifikasi","id_pengguna","jenis","judul","isi_pesan","dibaca","created_at"],
 "log_inferensi": ["id_log","id_pengguna","pertanyaan","intent","skor","jawaban","status","waktu"],
}
for t in COLS: T.setdefault(t, [])
ORDER = list(COLS)
FK = [("token_aktivasi","id_pengguna","pengguna","id_pengguna"),("barang","id_kategori","kategori_barang","id_kategori"),("stok_barang","kode_barang","barang","kode_barang"),
 ("stok_barang","id_gudang","gudang","id_gudang"),("transaksi_stok","id_pengguna","pengguna","id_pengguna"),("detail_transaksi_stok","id_transaksi","transaksi_stok","id_transaksi"),
 ("detail_transaksi_stok","kode_barang","barang","kode_barang"),("detail_transaksi_stok","id_gudang_asal","gudang","id_gudang"),("detail_transaksi_stok","id_gudang_tujuan","gudang","id_gudang"),
 ("riwayat_aktivitas","id_pengguna","pengguna","id_pengguna"),("riwayat_aktivitas","id_transaksi","transaksi_stok","id_transaksi"),("riwayat_aktivitas","kode_barang","barang","kode_barang"),
 ("riwayat_aktivitas","id_gudang","gudang","id_gudang"),("item_laporan","id_riwayat","riwayat_aktivitas","id_riwayat"),("item_laporan","diperiksa_oleh","pengguna","id_pengguna"),
 ("riwayat_validasi","id_item_laporan","item_laporan","id_item_laporan"),("riwayat_validasi","id_spv","pengguna","id_pengguna")]

def validasi():
    hasil = []
    def cek(nama, ok, detail=""): hasil.append((nama, "LULUS" if ok else "GAGAL", detail))
    # --- keterlacakan ke data mentah ---
    mentah_keluar = int(sales_raw.loc[sales_raw["Qty Sold"] > 0, "Qty Sold"].sum()); mentah_retur = int(-sales_raw.loc[sales_raw["Qty Sold"] < 0, "Qty Sold"].sum())
    cek("penjualan bersih: total keluar = data mentah", int(PENJUALAN.loc[(PENJUALAN.jenis == "keluar") & (PENJUALAN.sumber == "retail"), "qty"].sum()) == mentah_keluar, f"{mentah_keluar}")
    cek("penjualan bersih: total retur = data mentah", int(PENJUALAN.loc[(PENJUALAN.jenis == "retur") & (PENJUALAN.sumber == "retail"), "qty"].sum()) == mentah_retur, f"{mentah_retur}")
    mentah_aktif = int(inv_raw.loc[inv_raw["End Date"] == "9999-12-31", "Qty on hand"].clip(lower=0).sum())
    cek("stok aktif bersih = data mentah (stok negatif dijadikan 0)", int(AKTIF["qty"].sum()) == mentah_aktif, f"{mentah_aktif}")
    sku_keluar = int(PENJUALAN[PENJUALAN.kode_barang.isin(SKU) & (PENJUALAN.jenis == "keluar")]["qty"].sum()); sku_retur = int(PENJUALAN[PENJUALAN.kode_barang.isin(SKU) & (PENJUALAN.jenis == "retur")]["qty"].sum())
    rw = T["riwayat_aktivitas"]
    cek("kaos kaki: semua berasal dari Flipkart, nama memuat kata socks (bukan Socket)", len(SOCKS) > 0 and SOCKS["product_name"].str.contains(r"\bsocks?\b", case=False, regex=True).all() and not SOCKS["product_name"].str.contains("socket", case=False).any(), str(len(SOCKS)))
    cek("kaos kaki: penjualan dan stok berlabel sintetis", (PENJUALAN[PENJUALAN.kode_barang.isin(SOCKS.index)]["sumber"] == "sintetis_kaos_kaki").all() and all(BY_KODE[k]["sumber_data"].startswith("flipkart") for k in SOCKS.index))
    cek("riwayat 'keluar' = penjualan 300 produk", -sum(r["jumlah_perubahan"] for r in rw if r["jenis_aktivitas"] == "keluar") == sku_keluar, f"{sku_keluar}")
    cek("riwayat retur = retur 300 produk", sum(r["jumlah_perubahan"] for r in rw if r["alasan"].startswith("Retur")) == sku_retur, f"{sku_retur}")
    # --- aturan template ---
    cek("nama kolom setiap baris sesuai skema", all(set(r) <= set(COLS[t]) for t, rows in T.items() if t in COLS for r in rows))
    cek("jumlah produk aplikasi", len(BARANG) == len(SKU), str(len(SKU)))
    cek("kode unik, <=20 karakter, huruf/angka/strip", len({b['kode_barang'] for b in BARANG}) == len(BARANG) and all(len(b['kode_barang']) <= 20 and re.fullmatch(r"[A-Za-z0-9-]+", b['kode_barang']) for b in BARANG))
    cek("nama unik dan <=80 karakter", len({b['nama_barang'] for b in BARANG}) == len(BARANG) and all(len(b['nama_barang']) <= 80 for b in BARANG))
    cek("harga beli < harga jual", all(b['harga_beli'] < b['harga_jual'] for b in BARANG))
    cek("minimum_stok > 0 dan diskon 0..100", all(b['minimum_stok'] > 0 and 0 <= b['diskon'] <= 100 for b in BARANG))
    # --- stok & riwayat ---
    cek("stok >= 0", all(v >= 0 for v in SIS.values()))
    cek("stok_barang = snapshot inventory bersih (per barang-gudang)", all(r['jumlah_stok'] == STOK_F.get((r['kode_barang'], r['id_gudang']), 0) for r in T['stok_barang']))
    cek("total stok per produk = master bersih", all(TOTAL[k] == int(MASTER.loc[k, 'stok_total_bersih']) for k in SKU))
    jml = defaultdict(int); terakhir = {}; rantai = True
    for r in rw:
        key = (r['kode_barang'], r['id_gudang']); jml[key] += r['jumlah_perubahan']
        if r['jumlah_sebelum'] != terakhir.get(key, 0) or r['jumlah_sebelum'] + r['jumlah_perubahan'] != r['jumlah_sesudah']: rantai = False
        terakhir[key] = r['jumlah_sesudah']
    cek("riwayat: sebelum+perubahan=sesudah dan berantai", rantai)
    cek("riwayat: jumlah semua perubahan = stok akhir", all(jml.get((r['kode_barang'], r['id_gudang']), 0) == r['jumlah_stok'] for r in T['stok_barang']))
    cek("riwayat: waktu menaik sesuai id", all(a['waktu_aktivitas'] <= b['waktu_aktivitas'] for a, b in zip(rw, rw[1:])))
    cek("item_laporan sesuai riwayat_validasi", all(ITEM[v['id_item_laporan']]['status_item'] == v['status_baru'] for v in T['riwayat_validasi']))
    # --- Kasir ---
    cek("kunci jawaban Kasir konsisten dengan stok mentah/bersih", all((r['status_diharapkan'] == 'tidak_ditemukan') == (r['kode_barang'] not in BY_KODE) for r in df_exp.to_dict('records')))
    fk_bad = []
    for t, c, rt, rc in FK:
        ref = {r[rc] for r in T[rt]}
        if not all((r[c] is None) or (r[c] in ref) for r in T[t]): fk_bad.append(f"{t}.{c}")
    cek("semua foreign key valid", not fk_bad, ", ".join(fk_bad))
    # --- AI ---
    d = DF_INTENT; d["n"] = d["teks"].map(norm)
    cek("intent: >= TARGET_PER_INTENT kalimat per kelas", (d.groupby("intent").size() >= TARGET_PER_INTENT).all())
    cek("intent: tidak ada kalimat ganda", d["n"].is_unique)
    cek("intent: latih dan uji tidak beririsan (kalimat & template)", not (set(d[d.split == 'latih']['n']) & set(d[d.split == 'uji']['n'])) and not (set(d[d.split == 'latih']['template_id']) & set(d[d.split == 'uji']['template_id'])))
    cek("intent: hold-out di luar konteks tidak ada di dataset", not ({norm(x) for x in OOC_UJI} & set(d['n'])))
    df = pd.DataFrame(hasil, columns=["pemeriksaan", "hasil", "catatan"]); print(df.to_string(index=False))
    assert (df["hasil"] == "LULUS").all(), "Ada pemeriksaan GAGAL, periksa tabel di atas"
    return df
DF_VALIDASI = validasi()

                                                                  pemeriksaan hasil catatan
                                 penjualan bersih: total keluar = data mentah LULUS  124542
                                  penjualan bersih: total retur = data mentah LULUS    7547
                   stok aktif bersih = data mentah (stok negatif dijadikan 0) LULUS   37576
kaos kaki: semua berasal dari Flipkart, nama memuat kata socks (bukan Socket) LULUS      33
                              kaos kaki: penjualan dan stok berlabel sintetis LULUS        
                                      riwayat 'keluar' = penjualan 300 produk LULUS   24178
                                             riwayat retur = retur 300 produk LULUS    1280
                                         nama kolom setiap baris sesuai skema LULUS        
                                                       jumlah produk aplikasi LULUS     300
                                  kode unik, <=20 karakter, huruf/angka/strip LU

## 10. Ekspor CSV, SQL (dipecah per berkas), analisis, dan dokumentasi

In [10]:
# ============ 10. EKSPOR ============
DDL = """-- AkutuStocks | SKEMA SEMENTARA (PostgreSQL / Supabase) | disusun dari ERD konsep awal | DATA CONTOH
-- Jalankan file ini dulu, lalu berkas 01_*, 02_*, ... berurutan, terakhir 99_reset_sequences.sql. Ravina/Zahra dapat menyesuaikan setelah skema final.
DROP VIEW IF EXISTS v_stok_ringkas;
DROP TABLE IF EXISTS log_inferensi, notifikasi, tindak_lanjut_selisih, selisih_stok, perbandingan_stok, detail_import, import_kasir,
  riwayat_validasi, item_laporan, riwayat_aktivitas, detail_stok_opname, stock_opname, detail_transaksi_stok, transaksi_stok,
  stok_barang, barang, token_aktivasi, pengguna, kategori_barang, gudang CASCADE;
DROP FUNCTION IF EXISTS tolak_ubah_riwayat();

CREATE TABLE gudang (
  id_gudang INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  nama_gudang VARCHAR(100) NOT NULL UNIQUE);
CREATE TABLE kategori_barang (
  id_kategori INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  nama_kategori VARCHAR(100) NOT NULL UNIQUE, deskripsi VARCHAR(255), divisi VARCHAR(100));
CREATE TABLE pengguna (
  id_pengguna INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  nama_lengkap VARCHAR(100) NOT NULL, email VARCHAR(100) NOT NULL UNIQUE,
  password_hash VARCHAR(255),
  no_telepon VARCHAR(20) NOT NULL, alamat VARCHAR(255),
  status_akun VARCHAR(20) NOT NULL CHECK (status_akun IN ('belum_aktif','aktif','nonaktif')),
  role VARCHAR(10) NOT NULL CHECK (role IN ('owner','spv','staff')),
  wajib_ganti_password BOOLEAN NOT NULL DEFAULT FALSE,
  shift_kerja VARCHAR(30), nama_usaha VARCHAR(150), alamat_usaha VARCHAR(255),
  created_at TIMESTAMPTZ NOT NULL DEFAULT now(), updated_at TIMESTAMPTZ,
  CHECK (status_akun = 'belum_aktif' OR password_hash IS NOT NULL));
CREATE TABLE token_aktivasi (
  id_token INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_pengguna INTEGER NOT NULL REFERENCES pengguna(id_pengguna),
  token_hash VARCHAR(255) NOT NULL, kedaluwarsa_pada TIMESTAMPTZ NOT NULL, dipakai_pada TIMESTAMPTZ,
  created_at TIMESTAMPTZ NOT NULL DEFAULT now());
CREATE TABLE barang (
  kode_barang VARCHAR(20) PRIMARY KEY CHECK (kode_barang ~ '^[A-Za-z0-9-]+$'),
  nama_barang VARCHAR(80) NOT NULL,
  id_kategori INTEGER NOT NULL REFERENCES kategori_barang(id_kategori),
  harga_beli NUMERIC(15,2) NOT NULL, harga_jual NUMERIC(15,2) NOT NULL,
  minimum_stok INTEGER NOT NULL CHECK (minimum_stok >= 0),
  tipe_diskon SMALLINT NOT NULL DEFAULT 0 CHECK (tipe_diskon IN (0,1)),
  diskon NUMERIC(12,2) NOT NULL DEFAULT 0 CHECK (diskon >= 0),
  satuan VARCHAR(20) NOT NULL, berat_gram INTEGER, keterangan VARCHAR(255), gambar VARCHAR(255),
  sumber_data VARCHAR(50) NOT NULL DEFAULT 'sintetis',
  created_at TIMESTAMPTZ NOT NULL DEFAULT now(), updated_at TIMESTAMPTZ,
  CHECK (harga_beli < harga_jual));
CREATE TABLE stok_barang (
  id_stok INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  kode_barang VARCHAR(20) NOT NULL REFERENCES barang(kode_barang),
  id_gudang INTEGER NOT NULL REFERENCES gudang(id_gudang),
  jumlah_stok INTEGER NOT NULL CHECK (jumlah_stok >= 0),
  letak_rak VARCHAR(30) NOT NULL, tanggal_pengecekan_terakhir DATE,
  UNIQUE (kode_barang, id_gudang));
CREATE TABLE transaksi_stok (
  id_transaksi INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_pengguna INTEGER NOT NULL REFERENCES pengguna(id_pengguna),
  jenis_transaksi VARCHAR(20) NOT NULL CHECK (jenis_transaksi IN ('masuk','keluar','pindah','penyesuaian')),
  waktu_transaksi TIMESTAMPTZ NOT NULL,
  status_transaksi VARCHAR(20) NOT NULL CHECK (status_transaksi IN ('Selesai','Dibatalkan')),
  alasan VARCHAR(255) NOT NULL, alasan_pembatalan VARCHAR(255),
  CHECK (status_transaksi <> 'Dibatalkan' OR alasan_pembatalan IS NOT NULL));
-- jumlah: positif untuk masuk/keluar/pindah; bertanda (+/-) untuk penyesuaian. masuk/penyesuaian memakai id_gudang_tujuan,
-- keluar memakai id_gudang_asal, pindah memakai keduanya.
CREATE TABLE detail_transaksi_stok (
  id_detail_transaksi INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_transaksi INTEGER NOT NULL REFERENCES transaksi_stok(id_transaksi),
  kode_barang VARCHAR(20) NOT NULL REFERENCES barang(kode_barang),
  id_gudang_asal INTEGER REFERENCES gudang(id_gudang), id_gudang_tujuan INTEGER REFERENCES gudang(id_gudang),
  jumlah INTEGER NOT NULL CHECK (jumlah <> 0), stok_target INTEGER, no_batch VARCHAR(50), catatan VARCHAR(255),
  CHECK (id_gudang_asal IS NOT NULL OR id_gudang_tujuan IS NOT NULL));
CREATE TABLE stock_opname (
  id_opname INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_gudang INTEGER NOT NULL REFERENCES gudang(id_gudang), id_pengguna INTEGER NOT NULL REFERENCES pengguna(id_pengguna),
  waktu_opname TIMESTAMPTZ NOT NULL, status_opname VARCHAR(30) NOT NULL, catatan VARCHAR(255));
CREATE TABLE detail_stok_opname (
  id_detail_opname INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_opname INTEGER NOT NULL REFERENCES stock_opname(id_opname),
  kode_barang VARCHAR(20) NOT NULL REFERENCES barang(kode_barang),
  stok_sistem INTEGER NOT NULL CHECK (stok_sistem >= 0), stok_fisik INTEGER NOT NULL CHECK (stok_fisik >= 0),
  jumlah_selisih INTEGER NOT NULL, keterangan VARCHAR(255),
  UNIQUE (id_opname, kode_barang));
-- Riwayat bergaya commit. Untuk jenis 'stok_fisik': jumlah_sesudah = hasil hitung fisik, jumlah_perubahan = 0 (stok sistem tidak berubah).
CREATE TABLE riwayat_aktivitas (
  id_riwayat INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_pengguna INTEGER NOT NULL REFERENCES pengguna(id_pengguna),
  id_transaksi INTEGER REFERENCES transaksi_stok(id_transaksi),
  id_detail_opname INTEGER REFERENCES detail_stok_opname(id_detail_opname),
  kode_barang VARCHAR(20) NOT NULL REFERENCES barang(kode_barang),
  id_gudang INTEGER NOT NULL REFERENCES gudang(id_gudang),
  jenis_aktivitas VARCHAR(20) NOT NULL CHECK (jenis_aktivitas IN ('masuk','keluar','pindah','penyesuaian','stok_fisik')),
  waktu_aktivitas TIMESTAMPTZ NOT NULL,
  jumlah_sebelum INTEGER NOT NULL, jumlah_sesudah INTEGER NOT NULL, jumlah_perubahan INTEGER NOT NULL,
  alasan VARCHAR(255) NOT NULL,
  id_riwayat_ref INTEGER REFERENCES riwayat_aktivitas(id_riwayat),
  CHECK (id_transaksi IS NOT NULL OR id_detail_opname IS NOT NULL));
CREATE INDEX idx_riwayat_barang_waktu ON riwayat_aktivitas (kode_barang, waktu_aktivitas);
CREATE INDEX idx_riwayat_gudang ON riwayat_aktivitas (id_gudang);
CREATE FUNCTION tolak_ubah_riwayat() RETURNS trigger AS $$
BEGIN RAISE EXCEPTION 'riwayat_aktivitas bersifat hanya-baca (tidak boleh diubah atau dihapus)'; END; $$ LANGUAGE plpgsql;
CREATE TRIGGER trg_riwayat_hanya_baca BEFORE UPDATE OR DELETE ON riwayat_aktivitas
  FOR EACH ROW EXECUTE FUNCTION tolak_ubah_riwayat();
CREATE TABLE item_laporan (
  id_item_laporan INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_riwayat INTEGER NOT NULL UNIQUE REFERENCES riwayat_aktivitas(id_riwayat),
  status_item VARCHAR(20) NOT NULL CHECK (status_item IN ('Diajukan','Diterima','Ditolak')),
  diperiksa_oleh INTEGER REFERENCES pengguna(id_pengguna), diajukan_pada TIMESTAMPTZ NOT NULL);
CREATE INDEX idx_item_status ON item_laporan (status_item);
CREATE TABLE riwayat_validasi (
  id_riwayat_validasi INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_item_laporan INTEGER NOT NULL REFERENCES item_laporan(id_item_laporan),
  id_spv INTEGER NOT NULL REFERENCES pengguna(id_pengguna),
  status_sebelumnya VARCHAR(20) NOT NULL, status_baru VARCHAR(20) NOT NULL,
  tanggal_validasi TIMESTAMPTZ NOT NULL, catatan_validasi VARCHAR(255),
  CHECK (status_baru <> 'Ditolak' OR catatan_validasi IS NOT NULL));
CREATE TABLE import_kasir (
  id_import INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_spv INTEGER NOT NULL REFERENCES pengguna(id_pengguna),
  nama_file VARCHAR(100) NOT NULL, format_file VARCHAR(50) NOT NULL, tanggal_import TIMESTAMPTZ NOT NULL, keterangan VARCHAR(255));
CREATE TABLE detail_import (
  id_detail_import INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_import INTEGER NOT NULL REFERENCES import_kasir(id_import),
  nomor_baris INTEGER NOT NULL, kode_barang_berkas VARCHAR(100) NOT NULL, nama_barang_berkas VARCHAR(255), jumlah_berkas INTEGER NOT NULL,
  status_pencocokan VARCHAR(30) NOT NULL CHECK (status_pencocokan IN ('sesuai','beda_jumlah','tidak_ditemukan')),
  alasan_gagal VARCHAR(100));
-- selisih = stok_pembanding (Kasir Pintar) - stok_website (total semua gudang). kode_barang NULL bila tidak ditemukan.
CREATE TABLE perbandingan_stok (
  id_perbandingan_stok INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_detail_import INTEGER NOT NULL REFERENCES detail_import(id_detail_import),
  kode_barang VARCHAR(20) REFERENCES barang(kode_barang),
  jenis_sumber VARCHAR(30) NOT NULL, stok_website INTEGER NOT NULL, stok_pembanding INTEGER NOT NULL, selisih INTEGER NOT NULL,
  status_perbandingan VARCHAR(30) NOT NULL CHECK (status_perbandingan IN ('sesuai','beda_jumlah','tidak_ditemukan')),
  tanggal_perbandingan TIMESTAMPTZ NOT NULL);
CREATE TABLE selisih_stok (
  id_selisih_stok INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_perbandingan_stok INTEGER REFERENCES perbandingan_stok(id_perbandingan_stok),
  id_detail_opname INTEGER REFERENCES detail_stok_opname(id_detail_opname),
  jenis_selisih VARCHAR(30) NOT NULL CHECK (jenis_selisih IN ('kurang_fisik','lebih_fisik','beda_jumlah','tidak_ditemukan')),
  status_kasus VARCHAR(30) NOT NULL CHECK (status_kasus IN ('Belum dicek','Sedang dicek ulang','Menunggu validasi','Selesai')),
  keterangan VARCHAR(255), created_at TIMESTAMPTZ NOT NULL,
  CHECK ((id_perbandingan_stok IS NOT NULL)::int + (id_detail_opname IS NOT NULL)::int = 1));
CREATE TABLE tindak_lanjut_selisih (
  id_tindak_lanjut INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_selisih_stok INTEGER NOT NULL REFERENCES selisih_stok(id_selisih_stok),
  id_pengguna INTEGER NOT NULL REFERENCES pengguna(id_pengguna),
  jenis_tindakan VARCHAR(30) NOT NULL CHECK (jenis_tindakan IN ('hitung_ulang','isi_hasil_fisik','validasi','penyesuaian_stok','koreksi_kasir')),
  tanggal_tindak_lanjut TIMESTAMPTZ NOT NULL, catatan VARCHAR(255));
CREATE TABLE notifikasi (
  id_notifikasi INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_pengguna INTEGER NOT NULL REFERENCES pengguna(id_pengguna),
  jenis VARCHAR(20) NOT NULL CHECK (jenis IN ('penolakan','selisih_baru','pengingat')),
  judul VARCHAR(100) NOT NULL, isi_pesan TEXT NOT NULL, dibaca BOOLEAN NOT NULL DEFAULT FALSE, created_at TIMESTAMPTZ NOT NULL);
CREATE TABLE log_inferensi (
  id_log INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
  id_pengguna INTEGER REFERENCES pengguna(id_pengguna), pertanyaan TEXT NOT NULL, intent VARCHAR(50), skor NUMERIC(5,4), jawaban TEXT,
  status VARCHAR(20) NOT NULL CHECK (status IN ('berhasil','ditolak','gagal')), waktu TIMESTAMPTZ NOT NULL DEFAULT now());
-- Status stok dihitung (tidak disimpan): habis / rendah (< minimum_stok total semua gudang) / aman.
CREATE VIEW v_stok_ringkas AS
SELECT b.kode_barang, b.nama_barang, b.minimum_stok, COALESCE(SUM(s.jumlah_stok),0) AS total_stok,
  CASE WHEN COALESCE(SUM(s.jumlah_stok),0) = 0 THEN 'habis'
       WHEN COALESCE(SUM(s.jumlah_stok),0) < b.minimum_stok THEN 'rendah' ELSE 'aman' END AS status_stok
FROM barang b LEFT JOIN stok_barang s ON s.kode_barang = b.kode_barang
GROUP BY b.kode_barang, b.nama_barang, b.minimum_stok;
"""

PK = {"gudang":"id_gudang","kategori_barang":"id_kategori","pengguna":"id_pengguna","token_aktivasi":"id_token","stok_barang":"id_stok","transaksi_stok":"id_transaksi",
      "detail_transaksi_stok":"id_detail_transaksi","stock_opname":"id_opname","detail_stok_opname":"id_detail_opname","riwayat_aktivitas":"id_riwayat","item_laporan":"id_item_laporan",
      "riwayat_validasi":"id_riwayat_validasi","import_kasir":"id_import","detail_import":"id_detail_import","perbandingan_stok":"id_perbandingan_stok","selisih_stok":"id_selisih_stok",
      "tindak_lanjut_selisih":"id_tindak_lanjut","notifikasi":"id_notifikasi","log_inferensi":"id_log"}
def lit(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "NULL"
    if isinstance(v, (bool, np.bool_)): return "TRUE" if v else "FALSE"
    if isinstance(v, (int, np.integer)): return str(int(v))
    if isinstance(v, (float, np.floating)): return repr(float(v))
    if isinstance(v, datetime): return "'" + v.isoformat(sep=" ") + "'"
    if isinstance(v, date): return "'" + v.isoformat() + "'"
    return "'" + str(v).replace("'", "''") + "'"
def csv_val(v): return v.isoformat(sep=" ") if isinstance(v, datetime) else v

# ---- CSV per tabel (untuk impor lewat Table Editor / COPY) ----
for t in ORDER:
    pd.DataFrame([{c: csv_val(r.get(c)) for c in COLS[t]} for r in T[t]], columns=COLS[t]).convert_dtypes().to_csv(f"{OUT_DIR}/fs/csv/{t}.csv", index=False)

# ---- SQL: skema + seed dipecah agar tiap berkas muat dijalankan di editor SQL ----
open(f"{OUT_DIR}/fs/sql/00_schema.sql", "w", encoding="utf-8").write(DDL)
MAKS_BYTE = 900_000; no = 0
def tulis_seed(tabel_list, nama):
    """Satu atau beberapa berkas BEGIN..COMMIT; tiap INSERT maks 500 baris, tiap berkas ~<=900 KB."""
    global no
    bagian, buf, ukuran = 1, [], 0
    def flush():
        nonlocal bagian, buf, ukuran
        if not buf: return
        sfx = f"_bagian{bagian}" if (bagian > 1 or ukuran >= MAKS_BYTE) else ""
        fn = f"{OUT_DIR}/fs/sql/{no:02d}_seed_{nama}{sfx}.sql"
        open(fn, "w", encoding="utf-8").write("-- DATA CONTOH (hasil olahan dataset publik). Jalankan berurutan setelah 00_schema.sql.\nBEGIN;\n" + "".join(buf) + "COMMIT;\n")
        bagian += 1; buf = []; ukuran = 0
    for t in tabel_list:
        rows = T[t]
        for i in range(0, len(rows), 500):
            vals = ",\n".join("  (" + ", ".join(lit(r.get(c)) for c in COLS[t]) + ")" for r in rows[i:i + 500])
            stmt = f"INSERT INTO {t} ({', '.join(COLS[t])}) VALUES\n{vals};\n"
            buf.append(stmt); ukuran += len(stmt)
            if ukuran >= MAKS_BYTE: flush()
    flush()
no = 1; tulis_seed(["gudang", "kategori_barang", "pengguna", "token_aktivasi", "barang", "stok_barang"], "master")
for nm in ["transaksi_stok", "detail_transaksi_stok", "riwayat_aktivitas", "item_laporan", "riwayat_validasi"]:
    no += 1; tulis_seed([nm], nm)
with open(f"{OUT_DIR}/fs/sql/99_reset_sequences.sql", "w", encoding="utf-8") as f:
    for t, pk in PK.items(): f.write(f"SELECT setval(pg_get_serial_sequence('{t}','{pk}'), COALESCE((SELECT MAX({pk}) FROM {t}), 0) + 1, false);\n")

# ---- keluaran analisis (Ihsan / Rini) ----
MASTER.reset_index().to_csv(f"{OUT_DIR}/analisis/produk_master_lengkap.csv", index=False)
AKTIF.assign(tgl_cek=AKTIF["tgl_cek"].dt.date).to_csv(f"{OUT_DIR}/analisis/stok_aktif_per_toko.csv", index=False)
PENJUALAN.assign(tanggal=PENJUALAN["tanggal"].dt.date).to_csv(f"{OUT_DIR}/analisis/penjualan_harian_bersih.csv.gz", index=False, compression="gzip")
DF_TOKO.to_csv(f"{OUT_DIR}/analisis/toko_ke_gudang.csv", index=False)
DF_ACUAN.to_csv(f"{OUT_DIR}/analisis/pemetaan_flipkart_acuan.csv", index=False)
APP.reset_index()[["kode_barang", "divisi", "kategori", "subkategori", "supplier", "harga_beli_diimputasi", "unit_keluar", "unit_retur", "stok_total_mentah", "stok_total_bersih"]].to_csv(f"{OUT_DIR}/analisis/produk_aplikasi_atribut_tambahan.csv", index=False)

SOCKS.drop(columns=["gambar"]).assign(gambar_url=SOCKS["gambar"]).to_csv(f"{OUT_DIR}/analisis/produk_kaos_kaki_flipkart.csv", index=False)
PETA = [
 ("data_kode_barang", "retail_inventory/sales: Product No", "dipakai apa adanya"), ("data_nama_barang", "retail_inventory: Product Description", "nama segmen + kode segmen; kode PROD- dibuang; maks 80"),
 ("kategori", "retail_inventory: Product Category", "27 kategori; divisi disimpan di kategori_barang.divisi"),
 ("data_harga_beli", "retail_inventory: Stock Unit Cost Price", f"median status Full Price; harga <=0 dibuang; beli>=jual diimputasi; dikali {HARGA_SKALA} (ASUMSI mata uang)"),
 ("data_harga_jual", "retail_inventory: Stock Unit Selling Price", f"median status Full Price; dikali {HARGA_SKALA} (ASUMSI)"),
 ("data_stok", "retail_inventory: Qty on hand", "jumlah snapshot aktif (End Date 9999-12-31) semua toko; negatif dijadikan 0 (kecuali berkas Kasir: stok mentah)"),
 ("minimum_stok", "retail_sales: Qty Sold (turunan)", f"ceil(penjualan mingguan x {LEAD_WEEKS}), dibatasi 3..50"),
 ("tipe_diskon / diskon", "retail_inventory: Stock Status + rasio harga", "persen; hanya bila >=50% stok aktif berstatus markdown"),
 ("berat_dan_satuan / berat", "flipkart: product_specifications", "gram; hanya alas kaki bernilai wajar; selain itu kosong"),
 ("gambar", "flipkart: image", "URL pertama; dipasangkan ACAK per kelompok kategori; tautan 2016 mungkin sudah tidak aktif"),
 ("keterangan", "flipkart: description", "150 karakter awal + label 'Data contoh'"),
 ("letak_rak", "(sintetis)", "dibuat dari gudang + kategori + angka acak"), ("data_barang_jasa / data_show_toko", "(konstanta)", "0 / 0"),
 ("kaos kaki (33 produk)", "flipkart: product_name, retail_price, discounted_price, image, description", f"produk ASLI Flipkart (kata utuh 'socks'); harga INR x {KURS_INR_IDR} (ASUMSI); beli diimputasi; diskon dari harga diskon asli; STOK & PENJUALAN SINTETIS"),
 ("gudang", "retail: Store (40 toko)", "dikelompokkan menjadi 3 gudang, seimbang menurut total stok"),
 ("riwayat_aktivitas", "retail_sales", "Qty positif -> keluar; Qty negatif -> retur (masuk); saldo awal, penerimaan, penyesuaian diestimasi agar berujung di snapshot"),
 ("pengguna", "(sintetis)", "7 akun fiktif @contoh.test")]
pd.DataFrame(PETA, columns=["kolom_target", "sumber", "aturan"]).to_csv(f"{OUT_DIR}/dokumentasi/pemetaan_kolom_template.csv", index=False)
prof = []
for t in ORDER:
    df = pd.read_csv(f"{OUT_DIR}/fs/csv/{t}.csv")
    for c in df.columns:
        prof.append(dict(tabel=t, kolom=c, tipe_pandas=str(df[c].dtype), jumlah_baris=len(df), jumlah_kosong=int(df[c].isna().sum()), contoh_nilai=str(df[c].dropna().iloc[0])[:60] if df[c].notna().any() else ""))
pd.DataFrame(prof).to_csv(f"{OUT_DIR}/dokumentasi/profil_kolom.csv", index=False)
DF_VALIDASI.to_csv(f"{OUT_DIR}/dokumentasi/hasil_validasi.csv", index=False)

n_pend = sum(1 for r in T["item_laporan"] if r["status_item"] == "Diajukan")
README = f"""# AkutuStocks | Dataset (diolah dari dataset publik)

**PERHATIAN:** ini DATA CONTOH, bukan data Akutusocks. Tiga sumber: `retail_sales_ml_apl`, `retail_inventory_ml_apl`, dan `flipkart_com-ecommerce_sample`
(dataset publik; periksa lisensi masing-masing di halaman Kaggle sebelum publikasi). Temuan analisis dari data ini tidak boleh diklaim sebagai kondisi nyata Akutusocks.

## Struktur folder
- 01_gathering/, 02_assessing/, 03_cleaning/ : profil dataset, temuan kualitas, catatan setiap langkah pembersihan
- fs/csv/ , fs/sql/ : tabel database (Full-Stack). Jalankan sql/00_schema.sql, lalu 01_*, 02_*, ... berurutan, terakhir 99_reset_sequences.sql
- fs/kasir_pintar/ : barang_aplikasi_format_template.xlsx, kasir_benar (uji impor), uji negatif, hasil_diharapkan_pencocokan.csv
- fs/skenario_uji/ : skenario 5 tercatat / 3 tersedia fisik (selisih 2 barang)
- ai/ : dataset_intent_latih/uji/lengkap.csv, uji_di_luar_konteks_holdout.csv, kamus_intent.csv
- analisis/ : produk_master_lengkap (2.326 produk), stok_aktif_per_toko, penjualan_harian_bersih (hanya kuantitas), toko_ke_gudang, pemetaan_flipkart_acuan
- dokumentasi/ : pemetaan_kolom_template.csv, profil_kolom.csv, hasil_validasi.csv

## Ringkasan hasil
- Produk aplikasi: {len(SKU)} = {N_RETAIL} produk retail (dari {len(elig)} kandidat, bertingkat per kategori) + {len(SOCKS)} produk kaos kaki asli dari Flipkart. Master lengkap {len(MASTER)} produk di analisis/.
- Periode riwayat: {TGL_MIN} s.d. {TGL_MAX} (tanggal asli, tidak digeser). Riwayat: {len(T['riwayat_aktivitas'])} baris. Status Diajukan (menunggu SPV): {n_pend}.
- 40 toko dikelompokkan menjadi 3 gudang ({', '.join(GUDANG_NAMA)}), seimbang menurut stok.

## Asumsi dan keterbatasan (WAJIB dibaca)
1. Tema data retail adalah **sepatu dan pakaian anak**, bukan kaos kaki. Kaos kaki hanya {len(SOCKS)} produk asli dari Flipkart (pencarian kata "sock" menangkap 28 produk Phone Socket Holder yang dikeluarkan). **Stok dan penjualan kaos kaki SINTETIS** (parameter SOCK_RATE_MEDIAN, SOCK_RETUR_PROB); hanya nama, gambar, harga, dan diskon yang berasal dari Flipkart. Harga INR dikali {KURS_INR_IDR} sebagai ASUMSI kasar; harga beli diimputasi (Flipkart tidak punya biaya); ada outlier (mis. kaos kaki Rp 400 ribuan).
2. Harga produk retail dikali {HARGA_SKALA} sebagai ASUMSI agar bermagnitudo rupiah; dataset tidak menyebut mata uang.
3. Kolom keuangan penjualan (Sales Amount, Cogs) TIDAK dipakai (batasan project plan). Penjualan retail hanya dipakai untuk kuantitas.
4. Flipkart tidak punya kunci ke data retail. Gambar, keterangan, dan berat dipasangkan ACAK per kelompok kategori untuk produk retail (lihat analisis/pemetaan_flipkart_acuan.csv); produk kaos kaki memakai data aslinya sendiri. Tautan gambar tahun 2016 mungkin sudah mati.
5. Penerimaan barang tidak ada di data sumber. Stok awal, penerimaan, dan penyesuaian akhir DIESTIMASI agar seluruh riwayat berujung tepat di snapshot stok. {n_restock} penerimaan diestimasi; {n_adj} dari {len(SKU)*len(GUDANG_IDS)} pasangan barang-gudang punya snapshot lebih rendah daripada perhitungan dari penjualan (dicatat sebagai penyesuaian akhir, status Diajukan).
6. Berkas Kasir Pintar memakai stok MENTAH inventory (ada stok negatif), jadi selisih dengan website muncul alami dari data kotor. Format ekspor asli belum ada dari PIC; diasumsikan sama dengan TEMPLATE_BARANG.
7. Akun, riwayat validasi, dan letak rak adalah sintetis. Tidak ada contoh penolakan SPV, opname, maupun impor historis di seed; fitur itu diuji lewat aplikasi memakai berkas di fs/kasir_pintar.
8. Tanggal riwayat lama (2025-2026). Fitur pengingat sebaiknya memakai tanggal acuan = tanggal data terakhir ({TGL_MAX}), bukan tanggal hari ini.
9. Dataset intent: kalimat dibangkitkan dari template buatan tim dengan entitas dari data retail; data uji dipisah per template. Variasi kelas prosedur_stok terbatas (24 template).
10. Akun contoh: password `{DEMO_PASSWORD}` (hash bcrypt), email @contoh.test.
"""
open(f"{OUT_DIR}/README.md", "w", encoding="utf-8").write(README)
print("Berkas SQL:", sorted(os.listdir(f"{OUT_DIR}/fs/sql")))

Berkas SQL: ['00_schema.sql', '01_seed_master.sql', '02_seed_transaksi_stok.sql', '03_seed_detail_transaksi_stok_bagian1.sql', '03_seed_detail_transaksi_stok_bagian2.sql', '04_seed_riwayat_aktivitas_bagian1.sql', '04_seed_riwayat_aktivitas_bagian2.sql', '04_seed_riwayat_aktivitas_bagian3.sql', '04_seed_riwayat_aktivitas_bagian4.sql', '05_seed_item_laporan_bagian1.sql', '05_seed_item_laporan_bagian2.sql', '06_seed_riwayat_validasi_bagian1.sql', '06_seed_riwayat_validasi_bagian2.sql', '06_seed_riwayat_validasi_bagian3.sql', '99_reset_sequences.sql']


## 11. Ringkasan dan unduh ZIP

In [11]:
# ============ 11. RINGKASAN & UNDUH ============
print("Selisih alami Kasir vs website:", int((df_exp["status_diharapkan"] == "beda_jumlah").sum()), "barang berbeda,", int((df_exp["status_diharapkan"] == "tidak_ditemukan").sum()), "tidak ditemukan")
print("Produk menurut sumber:", {k: int(v) for k, v in APP["sumber_data"].value_counts().items()})
print("Item laporan:", pd.DataFrame(T["item_laporan"])["status_item"].value_counts().to_dict())
tot = pd.DataFrame([{"k": k, "t": TOTAL[k], "m": BY_KODE[k]["minimum_stok"]} for k in SKU]); print("Produk stok di bawah minimum:", int((tot.t < tot.m).sum()), "dari", len(tot))
shutil.make_archive(OUT_DIR, "zip", OUT_DIR); print("ZIP:", OUT_DIR + ".zip", round(os.path.getsize(OUT_DIR + ".zip") / 1e6, 1), "MB")
try:
    from google.colab import files
    files.download(OUT_DIR + ".zip")
except ImportError:
    pass

Selisih alami Kasir vs website: 34 barang berbeda, 5 tidak ditemukan
Produk menurut sumber: {'retail_inventory+flipkart_acuan': 267, 'flipkart_produk_asli+stok_penjualan_sintetis': 33}
Item laporan: {'Diterima': 23083, 'Diajukan': 478}
Produk stok di bawah minimum: 69 dari 300
ZIP: akutustocks_dataset.zip 2.7 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>